In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob

# --- CONFIG ---
DATASET_DIR = "./dataset_voice"
SAMPLE_RATE = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLES_PER_PERSON = 2# Povećaj ako imaš više snimki
EPOCHS = 50
INPUT_SIZE = 128 * 128 # Jer radiš interpolate na (128, 128)

# --- MODEL: Linearni Klasifikator ---
class SimpleLinearModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.flatten = nn.Flatten()
        self.classifier = nn.Linear(INPUT_SIZE, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        return self.classifier(x)

# --- TVOJE FUNKCIJE ZA LOAD ---
def load_audio(file_path):
    waveform, sr = torchaudio.load(file_path)
    if sr != SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(sr, SAMPLE_RATE)
        waveform = resampler(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    return waveform

def audio_to_mel(waveform):
    # n_mels je ovdje 64, ali interpolate to rastegne na 128 visine
    mel_transform = torchaudio.transforms.MelSpectrogram(
        SAMPLE_RATE, n_mels=64, n_fft=1024, hop_length=256
    )
    mel_spec = mel_transform(waveform)
    mel_spec = torchaudio.transforms.AmplitudeToDB()(mel_spec)
    
    # Interpolacija na 128x128
    mel_spec = torch.nn.functional.interpolate(
        mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False
    )
    return mel_spec.squeeze(0) 

# --- BUILD DATASET ---
X, y = [], []
person_folders = sorted([f for f in glob.glob(os.path.join(DATASET_DIR, "*")) if os.path.isdir(f)])
NUM_CLASSES = len(person_folders)

print(f"Učitavam podatke za {NUM_CLASSES} klase...")

for label, folder in enumerate(person_folders):
    audio_files = glob.glob(os.path.join(folder, "**/*.wav"), recursive=True)[:SAMPLES_PER_PERSON]
    for f in audio_files:
        waveform = load_audio(f)
        mel = audio_to_mel(waveform)
        X.append(mel)
        y.append(label)

if len(X) == 0:
    raise RuntimeError("Nema .wav datoteka! Provjeri DATASET_DIR.")

X = torch.stack(X).to(DEVICE) 
y = torch.tensor(y).to(DEVICE)

# --- TRAINING ---
model = SimpleLinearModel(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\nKrećem s treningom...")
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        acc = (outputs.argmax(1) == y).float().mean()
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {loss.item():.4f} | Acc: {acc.item()*100:.1f}%")

print("\nGotovo! Sad koristiš svoj load_audio i linearni model.")

Učitavam podatke za 40 klase...

Krećem s treningom...
Epoch  10/50 | Loss: 64.9542 | Acc: 81.2%
Epoch  20/50 | Loss: 14.9932 | Acc: 95.0%
Epoch  30/50 | Loss: 2.2166 | Acc: 98.8%
Epoch  40/50 | Loss: 0.0000 | Acc: 100.0%
Epoch  50/50 | Loss: 0.0000 | Acc: 100.0%

Gotovo! Sad koristiš svoj load_audio i linearni model.


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import random

# --- CONFIG ---
DATASET_DIR = "./dataset_voice"
SAMPLE_RATE = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLES_PER_PERSON = 2
EPOCHS = 20

class VoiceCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        # Nakon dva MaxPool2d(2), 128x128 postaje 32x32
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 32 * 32, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.fc(self.conv(x))

# --- UTILS ---
def load_audio(file_path):
    waveform, sr = torchaudio.load(file_path)
    if sr != SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(sr, SAMPLE_RATE)
        waveform = resampler(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    return waveform

def audio_to_mel(waveform):
    # Mel-spektrogram je standard u prepoznavanju govora (Davis & Mermelstein, 1980)
    mel_transform = torchaudio.transforms.MelSpectrogram(
        SAMPLE_RATE, n_mels=64, n_fft=1024, hop_length=256
    )
    mel_spec = mel_transform(waveform)
    mel_spec = torchaudio.transforms.AmplitudeToDB()(mel_spec)
    
    # Interpolacija osigurava fiksnu veličinu ulaza za FC slojeve
    mel_spec = torch.nn.functional.interpolate(
        mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False
    )
    return mel_spec.squeeze(0) # Vraća [1, 128, 128]

# --- BUILD DATASET ---
X, y = [], []
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
NUM_CLASSES = len(person_folders)

for label, folder in enumerate(person_folders):
    audio_files = glob.glob(os.path.join(folder, "**/*.wav"), recursive=True)[:SAMPLES_PER_PERSON]
    for f in audio_files:
        waveform = load_audio(f)
        mel = audio_to_mel(waveform)
        X.append(mel)
        y.append(label)

# ISPRAVAK: stack() već stvara batch dimenziju. 
# Ako mel ima [1, 128, 128], stack(X) daje [Batch, 1, 128, 128]
X = torch.stack(X).to(DEVICE) 
y = torch.tensor(y).to(DEVICE)

# --- TRAINING ---
model = VoiceCNN(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    
    acc = (outputs.argmax(1) == y).float().mean()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss.item():.4f} | Acc: {acc.item()*100:.2f}%")

# --- TEST ---
model.eval()
with torch.no_grad():
    test_folder = random.choice(person_folders)
    test_file = random.choice(glob.glob(os.path.join(test_folder, "**/*.wav"), recursive=True))
    test_mel = audio_to_mel(load_audio(test_file)).unsqueeze(0).to(DEVICE) # [1, 1, 128, 128]
    output = model(test_mel)
    print(f"Predviđeno: {output.argmax(1).item()} | Stvarno: {person_folders.index(test_folder)}")


Epoch 1/20 | Loss: 4.1985 | Acc: 2.50%
Epoch 2/20 | Loss: 27.7830 | Acc: 2.50%
Epoch 3/20 | Loss: 17.2797 | Acc: 2.50%
Epoch 4/20 | Loss: 16.1768 | Acc: 2.50%
Epoch 5/20 | Loss: 12.9827 | Acc: 5.00%
Epoch 6/20 | Loss: 11.1732 | Acc: 5.00%
Epoch 7/20 | Loss: 8.6370 | Acc: 7.50%
Epoch 8/20 | Loss: 6.8703 | Acc: 11.25%
Epoch 9/20 | Loss: 5.8300 | Acc: 11.25%
Epoch 10/20 | Loss: 4.7999 | Acc: 13.75%
Epoch 11/20 | Loss: 4.0408 | Acc: 10.00%
Epoch 12/20 | Loss: 3.6211 | Acc: 7.50%
Epoch 13/20 | Loss: 3.4105 | Acc: 10.00%
Epoch 14/20 | Loss: 3.6755 | Acc: 3.75%
Epoch 15/20 | Loss: 3.5066 | Acc: 7.50%
Epoch 16/20 | Loss: 3.5230 | Acc: 5.00%
Epoch 17/20 | Loss: 3.5182 | Acc: 3.75%
Epoch 18/20 | Loss: 3.4930 | Acc: 3.75%
Epoch 19/20 | Loss: 3.4532 | Acc: 6.25%
Epoch 20/20 | Loss: 3.4217 | Acc: 8.75%
Predviđeno: 19 | Stvarno: 20


In [10]:
# NISMO NAPRAVILI SPLIT [facepalm]
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import random
import time
from datetime import datetime, timedelta

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLES_PER_PERSON = 5 
EPOCHS = 50 
LEARNING_RATE = 0.001

# --- MODEL (ResNet-ish) ---
class VoiceResNetish(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )

        self.features = nn.Sequential(
            conv_block(1, 32),   
            conv_block(32, 64),  
            conv_block(64, 128), 
            conv_block(128, 256) 
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), 
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# --- UTILS ---
def load_and_preprocess(file_path):
    waveform, sr = torchaudio.load(file_path)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    
    mel_spec = torchaudio.transforms.MelSpectrogram(
        SAMPLE_RATE, n_mels=64, n_fft=1024, hop_length=256
    )(waveform)
    mel_spec = torchaudio.transforms.AmplitudeToDB()(mel_spec)
    
    mel_spec = torch.nn.functional.interpolate(
        mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False
    ).squeeze(0)
    
    mu, std = mel_spec.mean(), mel_spec.std()
    return (mel_spec - mu) / (std + 1e-7)

# --- INITIALIZATION ---
print(f"{'='*50}\n[INFO] Sustav za klasifikaciju glasova\n{'='*50}")
print(f"[*] Device: {DEVICE}")
print(f"[*] Vrijeme početka: {datetime.now().strftime('%H:%M:%S')}")

person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
NUM_CLASSES = len(person_folders)

X_list, y_list = [], []
start_load = time.time()

for label, folder in enumerate(person_folders):
    audio_files = glob.glob(os.path.join(folder, "**/*.wav"), recursive=True)[:SAMPLES_PER_PERSON]
    for f in audio_files:
        X_list.append(load_and_preprocess(f))
        y_list.append(label)

X = torch.stack(X_list).to(DEVICE)
y = torch.tensor(y_list).to(DEVICE)

load_duration = time.time() - start_load
print(f"[*] Dataset učitan: {len(X)} uzoraka | Vrijeme: {load_duration:.2f}s")
print(f"[*] Broj klasa (osoba): {NUM_CLASSES}")

model = VoiceResNetish(num_classes=NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"[*] Broj parametara modela: {total_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# --- TRAINING WITH ETA ---
print(f"\n{'='*20} TRENING {'='*20}")
start_train = time.time()

for epoch in range(EPOCHS):
    epoch_start_time = time.time()
    
    model.train()
    optimizer.zero_grad()
    
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    
    # Metrike
    with torch.no_grad():
        pred = outputs.argmax(1)
        acc = (pred == y).float().mean() * 100
        
    # Kalkulacija vremena i ETA
    epoch_duration = time.time() - epoch_start_time
    elapsed_time = time.time() - start_train
    avg_time_per_epoch = elapsed_time / (epoch + 1)
    remaining_epochs = EPOCHS - (epoch + 1)
    eta_seconds = remaining_epochs * avg_time_per_epoch
    eta_str = str(timedelta(seconds=int(eta_seconds)))

    # Robustni ispis progresa
    progress_bar = f"[{'#' * int((epoch+1)/(EPOCHS/20))}{'-' * (20 - int((epoch+1)/(EPOCHS/20)))}]"
    print(f"Epoch {epoch+1:03d}/{EPOCHS} {progress_bar} | Loss: {loss.item():.4f} | Acc: {acc:.2f}% | ETA: {eta_str} | {1/epoch_duration:.1f} it/s", end='\r')

    if (epoch + 1) == EPOCHS: print() # Novi red na kraju

total_train_time = str(timedelta(seconds=int(time.time() - start_train)))
print(f"{'='*50}\n[DONE] Trening završen u: {total_train_time}\n{'='*50}")

# --- TEST RANDOM SAMPLE ---
model.eval()
with torch.no_grad():
    test_idx = random.randint(0, len(X) - 1)
    sample_input = X[test_idx].unsqueeze(0)
    actual_label = y[test_idx].item()
    
    start_inf = time.time()
    output = model(sample_input)
    inference_time = (time.time() - start_inf) * 1000
    
    pred_label = output.argmax(1).item()
    confidence = torch.softmax(output, dim=1)[0][pred_label].item() * 100

    print(f"\n[INFERENCE TEST]")
    print(f"ID uzorka: {test_idx}")
    print(f"Stvarno:  {os.path.basename(person_folders[actual_label])}")
    print(f"Predviđeno: {os.path.basename(person_folders[pred_label])} ({confidence:.2f}% confidence)")
    print(f"Vrijeme inferencije: {inference_time:.2f}ms")
'''
==================================================
[INFO] Sustav za klasifikaciju glasova
==================================================
[*] Device: cpu
[*] Vrijeme početka: 13:29:56
[*] Dataset učitan: 200 uzoraka | Vrijeme: 2.76s
[*] Broj klasa (osoba): 40
[*] Broj parametara modela: 426,856

==================== TRENING ====================
Epoch 001/50 [--------------------] | Loss: 3.7164 | Acc: 3.00% | ETA: 0:01:55 |Epoch 002/50 [--------------------] | Loss: 3.6352 | Acc: 5.00% | ETA: 0:02:17 |Epoch 003/50 [#-------------------] | Loss: 3.5915 | Acc: 4.00% | ETA: 0:02:30 |Epoch 004/50 [#-------------------] | Loss: 3.5493 | Acc: 8.00% | ETA: 0:02:32 |Epoch 005/50 [##------------------] | Loss: 3.5116 | Acc: 7.00% | ETA: 0:02:36 |Epoch 006/50 [##------------------] | Loss: 3.4721 | Acc: 10.00% | ETA: 0:02:36 Epoch 007/50 [##------------------] | Loss: 3.4220 | Acc: 13.00% | ETA: 0:02:37 Epoch 008/50 [###-----------------] | Loss: 3.3783 | Acc: 10.00% | ETA: 0:02:36 Epoch 009/50 [###-----------------] | Loss: 3.3416 | Acc: 14.00% | ETA: 0:02:33 Epoch 010/50 [####----------------] | Loss: 3.2938 | Acc: 10.50% | ETA: 0:02:30 Epoch 011/50 [####----------------] | Loss: 3.2347 | Acc: 13.00% | ETA: 0:02:27 Epoch 012/50 [####----------------] | Loss: 3.1880 | Acc: 16.00% | ETA: 0:02:24 Epoch 013/50 [#####---------------] | Loss: 3.1361 | Acc: 15.50% | ETA: 0:02:20 Epoch 014/50 [#####---------------] | Loss: 3.0712 | Acc: 22.50% | ETA: 0:02:17 Epoch 015/50 [######--------------] | Loss: 3.0025 | Acc: 23.00% | ETA: 0:02:13 Epoch 016/50 [######--------------] | Loss: 2.9608 | Acc: 26.00% | ETA: 0:02:10 Epoch 017/50 [######--------------] | Loss: 2.9164 | Acc: 27.50% | ETA: 0:02:06 Epoch 018/50 [#######-------------] | Loss: 2.8617 | Acc: 30.00% | ETA: 0:02:02 Epoch 019/50 [#######-------------] | Loss: 2.7815 | Acc: 35.00% | ETA: 0:01:58 Epoch 020/50 [########------------] | Loss: 2.7449 | Acc: 36.00% | ETA: 0:01:54 Epoch 021/50 [########------------] | Loss: 2.6491 | Acc: 43.50% | ETA: 0:01:50 Epoch 022/50 [########------------] | Loss: 2.5787 | Acc: 43.50% | ETA: 0:01:46 Epoch 023/50 [#########-----------] | Loss: 2.5168 | Acc: 48.00% | ETA: 0:01:42 Epoch 024/50 [#########-----------] | Loss: 2.4545 | Acc: 51.50% | ETA: 0:01:38 Epoch 025/50 [##########----------] | Loss: 2.3605 | Acc: 52.00% | ETA: 0:01:34 Epoch 026/50 [##########----------] | Loss: 2.3074 | Acc: 61.50% | ETA: 0:01:30 Epoch 027/50 [##########----------] | Loss: 2.2540 | Acc: 60.00% | ETA: 0:01:27 Epoch 028/50 [###########---------] | Loss: 2.1551 | Acc: 64.50% | ETA: 0:01:23 Epoch 029/50 [###########---------] | Loss: 2.0913 | Acc: 65.50% | ETA: 0:01:19 Epoch 030/50 [############--------] | Loss: 2.0386 | Acc: 69.50% | ETA: 0:01:15 Epoch 031/50 [############--------] | Loss: 1.9374 | Acc: 72.00% | ETA: 0:01:11 Epoch 032/50 [############--------] | Loss: 1.8585 | Acc: 71.00% | ETA: 0:01:08 Epoch 033/50 [#############-------] | Loss: 1.7923 | Acc: 73.00% | ETA: 0:01:04 Epoch 034/50 [#############-------] | Loss: 1.7358 | Acc: 78.50% | ETA: 0:01:00 Epoch 035/50 [##############------] | Loss: 1.6550 | Acc: 77.00% | ETA: 0:00:56 Epoch 036/50 [##############------] | Loss: 1.5963 | Acc: 82.00% | ETA: 0:00:52 Epoch 037/50 [##############------] | Loss: 1.4871 | Acc: 87.00% | ETA: 0:00:48 Epoch 038/50 [###############-----] | Loss: 1.4807 | Acc: 83.00% | ETA: 0:00:45 Epoch 039/50 [###############-----] | Loss: 1.3562 | Acc: 85.50% | ETA: 0:00:41 Epoch 040/50 [################----] | Loss: 1.3329 | Acc: 84.50% | ETA: 0:00:37 Epoch 041/50 [################----] | Loss: 1.2510 | Acc: 85.50% | ETA: 0:00:33 Epoch 042/50 [################----] | Loss: 1.2041 | Acc: 88.00% | ETA: 0:00:30 Epoch 043/50 [#################---] | Loss: 1.1408 | Acc: 90.50% | ETA: 0:00:26 Epoch 044/50 [#################---] | Loss: 1.1159 | Acc: 86.50% | ETA: 0:00:22 Epoch 045/50 [##################--] | Loss: 1.0008 | Acc: 91.00% | ETA: 0:00:18 Epoch 046/50 [##################--] | Loss: 0.9149 | Acc: 93.00% | ETA: 0:00:15 Epoch 047/50 [##################--] | Loss: 0.8851 | Acc: 92.00% | ETA: 0:00:11 Epoch 048/50 [###################-] | Loss: 0.7997 | Acc: 94.00% | ETA: 0:00:07 Epoch 049/50 [###################-] | Loss: 0.7989 | Acc: 94.50% | ETA: 0:00:03 Epoch 050/50 [####################] | Loss: 0.7481 | Acc: 94.50% | ETA: 0:00:00 | 0.3 it/s
==================================================
[DONE] Trening završen u: 0:03:07
==================================================

[INFERENCE TEST]
ID uzorka: 135
Stvarno:  id10297
Predviđeno: id10297 (33.90% confidence)
Vrijeme inferencije: 16.75ms
'''

[INFO] Sustav za klasifikaciju glasova
[*] Device: cpu
[*] Vrijeme početka: 11:30:33
[*] Dataset učitan: 200 uzoraka | Vrijeme: 9.90s
[*] Broj klasa (osoba): 40
[*] Broj parametara modela: 426,856

==================== TRENING ====================
Epoch 050/50 [####################] | Loss: 0.7274 | Acc: 95.50% | ETA: 0:00:00 | 0.2 it/s
[DONE] Trening završen u: 0:04:31

[INFERENCE TEST]
ID uzorka: 52
Stvarno:  id10280
Predviđeno: id10270 (11.08% confidence)
Vrijeme inferencije: 13.64ms


'\n==================================================\n[INFO] Sustav za klasifikaciju glasova\n==================================================\n[*] Device: cpu\n[*] Vrijeme početka: 13:29:56\n[*] Dataset učitan: 200 uzoraka | Vrijeme: 2.76s\n[*] Broj klasa (osoba): 40\n[*] Broj parametara modela: 426,856\n\n==================== TRENING ====================\nEpoch 001/50 [--------------------] | Loss: 3.7164 | Acc: 3.00% | ETA: 0:01:55 |Epoch 002/50 [--------------------] | Loss: 3.6352 | Acc: 5.00% | ETA: 0:02:17 |Epoch 003/50 [#-------------------] | Loss: 3.5915 | Acc: 4.00% | ETA: 0:02:30 |Epoch 004/50 [#-------------------] | Loss: 3.5493 | Acc: 8.00% | ETA: 0:02:32 |Epoch 005/50 [##------------------] | Loss: 3.5116 | Acc: 7.00% | ETA: 0:02:36 |Epoch 006/50 [##------------------] | Loss: 3.4721 | Acc: 10.00% | ETA: 0:02:36 Epoch 007/50 [##------------------] | Loss: 3.4220 | Acc: 13.00% | ETA: 0:02:37 Epoch 008/50 [###-----------------] | Loss: 3.3783 | Acc: 10.00% | ETA: 0:02:

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import time
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.0005

# --- MODEL (ResNet-ish) ---
class VoiceResNetish(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )
        self.features = nn.Sequential(
            conv_block(1,32), conv_block(32,64),
            conv_block(64,128), conv_block(128,256)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,num_classes)
        )
    def forward(self,x):
        return self.classifier(self.features(x))

# --- PREPROCESS ---
def load_and_preprocess(file_path):
    waveform, sr = torchaudio.load(file_path)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    mel_spec = torchaudio.transforms.MelSpectrogram(SAMPLE_RATE, n_mels=64)(waveform)
    mel_spec = torchaudio.transforms.AmplitudeToDB()(mel_spec)
    mel_spec = torch.nn.functional.interpolate(
        mel_spec.unsqueeze(0), size=(128,128), mode='bilinear', align_corners=False
    ).squeeze(0)
    return (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-7)

# --- PREPARE DATA ---
print(f"--- START: {datetime.now().strftime('%H:%M:%S')} ---")
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
class_names = [os.path.basename(f) for f in person_folders]

# --- AUTOMATSKI SAMPLES_PER_PERSON ---
samples_per_person = min(len(glob.glob(os.path.join(f, "**/*.wav"), recursive=True)) for f in person_folders)
print(f"[DATA] Broj uzoraka po govorniku: {samples_per_person}")

X_all, y_all = [], []
for label, folder in enumerate(person_folders):
    files = glob.glob(os.path.join(folder, "**/*.wav"), recursive=True)[:samples_per_person]
    for f in files:
        X_all.append(load_and_preprocess(f))
        y_all.append(label)

X_all = torch.stack(X_all)
y_all = torch.tensor(y_all)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"[DATA] Trening: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

model = VoiceResNetish(len(class_names)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.02)
criterion = nn.CrossEntropyLoss()

best_val_loss = float('inf')

print(f"[INIT] Model spreman na {DEVICE}. Krećem s treningom...\n")

# --- TRAINING LOOP ---
start_train_time = time.time()
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        correct += (outputs.argmax(1) == batch_y).sum().item()
        total += batch_x.size(0)
    train_loss = running_loss / total
    train_acc = correct / total * 100

    # --- VAL ---
    model.eval()
    running_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            running_loss += loss.item() * batch_x.size(0)
            correct += (outputs.argmax(1) == batch_y).sum().item()
            total += batch_x.size(0)
    val_loss = running_loss / total
    val_acc = correct / total * 100

    print(f"Epoch {epoch+1:03d}/{EPOCHS} | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_voice_model.pth")

# --- FINAL EVAL ---
print(f"\n[DONE] Trening gotov. Ukupno vrijeme: {str(timedelta(seconds=int(time.time() - start_train_time)))}")
model.load_state_dict(torch.load("best_voice_model.pth"))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(DEVICE)
        outputs = model(batch_x)
        preds = outputs.argmax(1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(batch_y.numpy())

print("\n" + "="*40)
print(" FINALNI IZVJEŠTAJ (TEST SET) ")
print("="*40)
print(classification_report(y_true, y_pred, target_names=class_names))

--- START: 11:37:04 ---
[DATA] Broj uzoraka po govorniku: 48
[DATA] Trening: 1536 | Val: 192 | Test: 192
[INIT] Model spreman na cpu. Krećem s treningom...

Epoch 001/100 | Train Loss: 3.6349, Train Acc: 5.21% | Val Loss: 3.5248, Val Acc: 7.81%
Epoch 002/100 | Train Loss: 3.3769, Train Acc: 8.98% | Val Loss: 3.2604, Val Acc: 12.50%
Epoch 003/100 | Train Loss: 3.1073, Train Acc: 12.70% | Val Loss: 3.2083, Val Acc: 11.98%
Epoch 004/100 | Train Loss: 2.8615, Train Acc: 18.23% | Val Loss: 3.9330, Val Acc: 6.77%
Epoch 005/100 | Train Loss: 2.6761, Train Acc: 24.28% | Val Loss: 2.6804, Val Acc: 20.31%
Epoch 006/100 | Train Loss: 2.5064, Train Acc: 29.10% | Val Loss: 2.9721, Val Acc: 18.75%
Epoch 007/100 | Train Loss: 2.3612, Train Acc: 31.45% | Val Loss: 2.5109, Val Acc: 28.12%
Epoch 008/100 | Train Loss: 2.2083, Train Acc: 37.24% | Val Loss: 2.6083, Val Acc: 23.96%
Epoch 009/100 | Train Loss: 2.1039, Train Acc: 39.71% | Val Loss: 2.2705, Val Acc: 34.38%
Epoch 010/100 | Train Loss: 1.9865, T

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import time
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm # Import za progress bar

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.05

# --- MODEL (VoiceNet Slim) ---
class VoiceNetSlim(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.Dropout2d(0.15),
                nn.MaxPool2d(2)
            )
        self.features = nn.Sequential(
            conv_block(1, 16), conv_block(16, 32), conv_block(32, 64)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.6),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# --- DATASET ---
class VoiceDataset(Dataset):
    def __init__(self, file_paths, labels, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=35)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = self.labels[idx]
        waveform, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(0, keepdim=True)
        if self.augment:
            waveform = waveform + 0.005 * torch.randn_like(waveform)
        mel_spec = torchaudio.transforms.MelSpectrogram(SAMPLE_RATE, n_mels=64)(waveform)
        mel_spec = torchaudio.transforms.AmplitudeToDB()(mel_spec)
        if self.augment:
            mel_spec = self.freq_mask(mel_spec)
            mel_spec = self.time_mask(mel_spec)
        mel_spec = torch.nn.functional.interpolate(mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False).squeeze(0)
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-7)
        return mel_spec, label

# --- DATA PREP ---
print(f"--- START: {datetime.now().strftime('%H:%M:%S')} ---")
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
class_names = [os.path.basename(f) for f in person_folders]
all_files, all_labels = [], []
for label, folder in enumerate(person_folders):
    files = glob.glob(os.path.join(folder, "**/*.wav"), recursive=True)
    for f in files:
        all_files.append(f)
        all_labels.append(label)

X_train_f, X_temp_f, y_train, y_temp = train_test_split(all_files, all_labels, test_size=0.2, stratify=all_labels, random_state=42)
X_val_f, X_test_f, y_val, y_test = train_test_split(X_temp_f, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_loader = DataLoader(VoiceDataset(X_train_f, y_train, augment=True), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(VoiceDataset(X_val_f, y_val, augment=False), batch_size=BATCH_SIZE)
test_loader = DataLoader(VoiceDataset(X_test_f, y_test, augment=False), batch_size=BATCH_SIZE)

# --- SETUP ---
model = VoiceNetSlim(len(class_names)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

best_val_loss = float('inf')

# --- TRAINING LOOP ---
print(f"[SYSTEM] Trening na: {DEVICE}\n")

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_total = 0, 0, 0
    
    # TQDM progress bar za trening batch-eve
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [TRAIN]", unit="batch", leave=False)
    
    for bx, by in train_pbar:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * bx.size(0)
        t_corr += (out.argmax(1) == by).sum().item()
        t_total += bx.size(0)
        
        # Update progress bar info
        train_pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{100*t_corr/t_total:.2f}%"})
    
    # --- VALIDATION ---
    model.eval()
    v_loss, v_corr, v_total = 0, 0, 0
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [VAL]", unit="batch", leave=False)
    
    with torch.no_grad():
        for bx, by in val_pbar:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            out = model(bx)
            loss = criterion(out, by)
            v_loss += loss.item() * bx.size(0)
            v_corr += (out.argmax(1) == by).sum().item()
            v_total += bx.size(0)
            val_pbar.set_postfix({"v_loss": f"{loss.item():.4f}", "v_acc": f"{100*v_corr/v_total:.2f}%"})

    train_l, train_a = t_loss/t_total, t_corr/t_total*100
    val_l, val_a = v_loss/v_total, v_corr/v_total*100
    current_lr = optimizer.param_groups[0]['lr']
    
    scheduler.step(val_l)
    
    # Finalni ispis za epohu nakon što se pbar-ovi maknu
    print(f"Epoch {epoch+1:03d} | L: {train_l:.4f}, A: {train_a:.2f}% | VL: {val_l:.4f}, VA: {val_a:.2f}% | LR: {current_lr:.6f}")

    if val_l < best_val_loss:
        best_val_loss = val_l
        torch.save(model.state_dict(), "best_slim_model.pth")
        print(f"  [+] Saved best model (Loss: {best_val_loss:.4f})")

# --- FINAL EVAL ---
print(f"\n[DONE] Trening završen. Učitavam najbolji model za testiranje...")
model.load_state_dict(torch.load("best_slim_model.pth"))
model.eval()
y_true, y_pred = [], []
for bx, by in tqdm(test_loader, desc="Testing"):
    with torch.no_grad():
        out = model(bx.to(DEVICE))
        y_pred.extend(out.argmax(1).cpu().numpy())
        y_true.extend(by.numpy())

print("\n" + "="*40 + "\n FINALNI IZVJEŠTAJ \n" + "="*40)
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# best so far
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import time
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 80
LEARNING_RATE = 0.001  # Brže učenje za početak
WEIGHT_DECAY = 0.01    # Blaža regularizacija

# --- MODEL (VoiceNet Medium - Pojačana verzija) ---
class VoiceNetMedium(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )
        
        # Proširena arhitektura: 32 -> 64 -> 128 -> 128
        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 128)
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.4), # Smanjen dropout da ne "guši" učenje
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# --- DATASET KLASA ---
class VoiceDataset(Dataset):
    def __init__(self, file_paths, labels, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment
        # SpecAugment parametri
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=20)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=40)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = self.labels[idx]
        
        waveform, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(0, keepdim=True)
            
        # Dodavanje šuma samo u treningu
        if self.augment:
            waveform = waveform + 0.002 * torch.randn_like(waveform)

        # Povećan broj melsa na 128 za bolju rezoluciju glasa
        mel_spec = torchaudio.transforms.MelSpectrogram(SAMPLE_RATE, n_mels=128)(waveform)
        mel_spec = torchaudio.transforms.AmplitudeToDB()(mel_spec)
        
        if self.augment:
            mel_spec = self.freq_mask(mel_spec)
            mel_spec = self.time_mask(mel_spec)
            
        # Standardizacija veličine slike na 128x128
        mel_spec = torch.nn.functional.interpolate(
            mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False
        ).squeeze(0)
        
        # Normalizacija spektrograma
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-7)
        return mel_spec, label

# --- PRIPREMA PODATAKA ---
print(f"--- START: {datetime.now().strftime('%H:%M:%S')} ---")
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
class_names = [os.path.basename(f) for f in person_folders]

all_files, all_labels = [], []
for label, folder in enumerate(person_folders):
    files = glob.glob(os.path.join(folder, "**/*.wav"), recursive=True)
    for f in files:
        all_files.append(f)
        all_labels.append(label)

X_train_f, X_temp_f, y_train, y_temp = train_test_split(all_files, all_labels, test_size=0.2, stratify=all_labels, random_state=42)
X_val_f, X_test_f, y_val, y_test = train_test_split(X_temp_f, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_loader = DataLoader(VoiceDataset(X_train_f, y_train, augment=True), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(VoiceDataset(X_val_f, y_val, augment=False), batch_size=BATCH_SIZE)
test_loader = DataLoader(VoiceDataset(X_test_f, y_test, augment=False), batch_size=BATCH_SIZE)

print(f"[DATA] Klasa: {len(class_names)} | Train: {len(X_train_f)} | Val: {len(X_val_f)}")

# --- TRENING SETUP ---
model = VoiceNetMedium(len(class_names)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05) # Smanjen smoothing
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=7, factor=0.5)

best_val_loss = float('inf')

# --- LOOP ---
for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_total = 0, 0, 0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [TRAIN]", unit="batch", leave=False)
    
    for bx, by in train_pbar:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * bx.size(0)
        t_corr += (out.argmax(1) == by).sum().item()
        t_total += bx.size(0)
        train_pbar.set_postfix({"loss": f"{loss.item():.3f}", "acc": f"{100*t_corr/t_total:.1f}%"})
    
    model.eval()
    v_loss, v_corr, v_total = 0, 0, 0
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [VAL]", unit="batch", leave=False)
    
    with torch.no_grad():
        for bx, by in val_pbar:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            out = model(bx)
            loss = criterion(out, by)
            v_loss += loss.item() * bx.size(0)
            v_corr += (out.argmax(1) == by).sum().item()
            v_total += bx.size(0)
            val_pbar.set_postfix({"v_loss": f"{loss.item():.3f}", "v_acc": f"{100*v_corr/v_total:.1f}%"})

    train_l, train_a = t_loss/t_total, t_corr/t_total*100
    val_l, val_a = v_loss/v_total, v_corr/v_total*100
    
    scheduler.step(val_l)
    
    print(f"Epoch {epoch+1:03d} | L: {train_l:.4f}, A: {train_a:.2f}% | VL: {val_l:.4f}, VA: {val_a:.2f}% | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if val_l < best_val_loss:
        best_val_loss = val_l
        torch.save(model.state_dict(), "best_medium_model.pth")
        print(f"  [+] New Best Model: {best_val_loss:.4f}")

# --- EVALUACIJA ---
print(f"\n[TESTING] Učitavam najbolji model...")
model.load_state_dict(torch.load("best_medium_model.pth"))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for bx, by in tqdm(test_loader, desc="Final Eval"):
        out = model(bx.to(DEVICE))
        y_pred.extend(out.argmax(1).cpu().numpy())
        y_true.extend(by.numpy())

print("\n" + "="*40 + "\n IZVJEŠTAJ \n" + "="*40)
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 80 # 80
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01
EARLY_STOP_PATIENCE = 15 

# --- LOGGING SETUP ---
def get_run_folder(base_name="run"):
    n = 1
    while os.path.exists(f"{base_name}_{n}"):
        n += 1
    run_dir = f"{base_name}_{n}"
    os.makedirs(run_dir)
    return run_dir

RUN_DIR = get_run_folder()
print(f"\n🚀 START: Svi rezultati idu u folder: {RUN_DIR}")

# --- MODEL (VoiceNet Medium) ---
class VoiceNetMedium(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )
        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 128)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# --- DATASET KLASA ---
class VoiceDataset(Dataset):
    def __init__(self, file_paths, labels, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment
        self.freq_mask = torchaudio.transforms.FrequencyMasking(20)
        self.time_mask = torchaudio.transforms.TimeMasking(40)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = self.labels[idx]
        waveform, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(0, keepdim=True)
        
        if self.augment:
            waveform = waveform + 0.002 * torch.randn_like(waveform)
            
        mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, 
            n_mels=128, 
            n_fft=1024, 
            hop_length=512
        )(waveform)
        
        mel_spec = torchaudio.transforms.AmplitudeToDB()(mel_spec)
        
        if self.augment:
            mel_spec = self.freq_mask(mel_spec)
            mel_spec = self.time_mask(mel_spec)
            
        mel_spec = torch.nn.functional.interpolate(
            mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False
        ).squeeze(0)
        
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-7)
        return mel_spec, label

# --- PRIPREMA PODATAKA ---
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
class_names = [os.path.basename(f) for f in person_folders]

config_data = {
    "run_date": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "lr": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "num_classes": len(class_names),
    "early_stop_patience": EARLY_STOP_PATIENCE
}
with open(os.path.join(RUN_DIR, "config.json"), "w") as f:
    json.dump(config_data, f, indent=4)

all_files, all_labels = [], []
for label, folder in enumerate(person_folders):
    files = glob.glob(os.path.join(folder, "**/*.wav"), recursive=True)
    for f in files:
        all_files.append(f)
        all_labels.append(label)

X_train_f, X_temp_f, y_train, y_temp = train_test_split(all_files, all_labels, test_size=0.2, stratify=all_labels, random_state=42)
X_val_f, X_test_f, y_val, y_test = train_test_split(X_temp_f, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_loader = DataLoader(VoiceDataset(X_train_f, y_train, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(VoiceDataset(X_val_f, y_val, augment=False), batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(VoiceDataset(X_test_f, y_test, augment=False), batch_size=BATCH_SIZE)

# --- TRENING SETUP ---
model = VoiceNetMedium(len(class_names)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=7, factor=0.5)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
patience_counter = 0

# --- TRENING LOOP ---
print(f"Uređaj za trening: {DEVICE}")
print("-" * 30)

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}", leave=False)
    
    for bx, by in pbar:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * bx.size(0)
        t_corr += (out.argmax(1) == by).sum().item()
        t_total += bx.size(0)
    
    model.eval()
    v_loss, v_corr, v_total = 0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            out = model(bx)
            loss = criterion(out, by)
            v_loss += loss.item() * bx.size(0)
            v_corr += (out.argmax(1) == by).sum().item()
            v_total += bx.size(0)

    # Izračun metrika
    curr_train_loss = t_loss / t_total
    curr_val_loss = v_loss / v_total
    curr_train_acc = 100 * t_corr / t_total
    curr_val_acc = 100 * v_corr / v_total

    history['train_loss'].append(curr_train_loss)
    history['val_loss'].append(curr_val_loss)
    history['train_acc'].append(curr_train_acc)
    history['val_acc'].append(curr_val_acc)
    
    scheduler.step(curr_val_loss)
    
    # ISPRAVLJEN PRINT: Dodan Train Acc za bolji nadzor
    print(f"Epoch {epoch+1:03d} | Train Acc: {curr_train_acc:>5.1f}% | Val Acc: {curr_val_acc:>5.1f}% | Train Loss: {curr_train_loss:.3f} | Val Loss: {curr_val_loss:.3f}")

    if curr_val_loss < best_val_loss:
        best_val_loss = curr_val_loss
        torch.save(model.state_dict(), os.path.join(RUN_DIR, "best_model.pth"))
        patience_counter = 0
    else:
        patience_counter += 1
        
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\n[!] EARLY STOPPING aktiviran na epohi {epoch+1}")
        break

# --- FINALNI IZVJEŠTAJI (Identično kao prije) ---
def save_reports():
    print("\nGeneriram finalne grafikone i izvještaje...")
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Val')
    plt.title('Loss')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train')
    plt.plot(history['val_acc'], label='Val')
    plt.title('Accuracy (%)')
    plt.legend()
    plt.savefig(os.path.join(RUN_DIR, "training_curves.png"))

    sample_spec, _ = next(iter(val_loader))
    plt.figure(figsize=(6, 4))
    #plt.imshow(sample_spec[0].cpu().numpy(), aspect='auto', origin='lower')
    plt.imshow(sample_spec[0].squeeze().cpu().numpy(), aspect='auto', origin='lower')
    plt.colorbar(format='%+2.0f dB')
    plt.title("Model Input Sample")
    plt.savefig(os.path.join(RUN_DIR, "input_sample.png"))

    model.load_state_dict(torch.load(os.path.join(RUN_DIR, "best_model.pth")))
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            out = model(bx.to(DEVICE))
            y_pred.extend(out.argmax(1).cpu().numpy())
            y_true.extend(by.numpy())

    with open(os.path.join(RUN_DIR, "report.txt"), "w") as f:
        f.write(f"Best Val Loss: {best_val_loss:.4f}\n")
        f.write("-" * 30 + "\n")
        f.write(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-7)
    plt.figure(figsize=(18, 14))
    sns.heatmap(cm_norm, annot=False, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Normalized Confusion Matrix')
    plt.savefig(os.path.join(RUN_DIR, "confusion_matrix.png"))
    plt.close('all')

save_reports()
print(f"✅ Sve završeno. Rezultati u: {RUN_DIR}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 80
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01
EARLY_STOP_PATIENCE = 15

# --- LOGGING SETUP (Automatski folderi za svaki experiment) ---
def get_run_folder(base_name="run"):
    n = 1
    while os.path.exists(f"{base_name}_{n}"):
        n += 1
    run_dir = f"{base_name}_{n}"
    os.makedirs(run_dir)
    return run_dir

RUN_DIR = get_run_folder()
print(f"\n🚀 Svi rezultati ovog treninga idu u: {RUN_DIR}")

# --- MODEL ---
class VoiceNetMedium(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )
        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 128)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# --- DATASET KLASA ---
class VoiceDataset(Dataset):
    def __init__(self, file_paths, labels, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment
        self.freq_mask = torchaudio.transforms.FrequencyMasking(20)
        self.time_mask = torchaudio.transforms.TimeMasking(40)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = self.labels[idx]
        waveform, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(0, keepdim=True)
        if self.augment:
            waveform = waveform + 0.002 * torch.randn_like(waveform)
            
        mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        )(waveform)
        mel_spec = torchaudio.transforms.AmplitudeToDB()(mel_spec)
        
        if self.augment:
            mel_spec = self.freq_mask(mel_spec)
            mel_spec = self.time_mask(mel_spec)
            
        mel_spec = torch.nn.functional.interpolate(
            mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False
        ).squeeze(0)
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-7)
        return mel_spec, label

# --- PRIPREMA ---
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
class_names = [os.path.basename(f) for f in person_folders]

# Logiranje konfiguracije
with open(os.path.join(RUN_DIR, "config.json"), "w") as f:
    json.dump({
        "lr": LEARNING_RATE, "batch_size": BATCH_SIZE, "epochs": EPOCHS, 
        "weight_decay": WEIGHT_DECAY, "num_classes": len(class_names)
    }, f, indent=4)

all_files, all_labels = [], []
for label, folder in enumerate(person_folders):
    files = glob.glob(os.path.join(folder, "**/*.wav"), recursive=True)
    for f in files:
        all_files.append(f)
        all_labels.append(label)

X_train_f, X_temp_f, y_train, y_temp = train_test_split(all_files, all_labels, test_size=0.2, stratify=all_labels, random_state=42)
X_val_f, X_test_f, y_val, y_test = train_test_split(X_temp_f, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_loader = DataLoader(VoiceDataset(X_train_f, y_train, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(VoiceDataset(X_val_f, y_val, augment=False), batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(VoiceDataset(X_test_f, y_test, augment=False), batch_size=BATCH_SIZE)

# --- TRENING SETUP ---
model = VoiceNetMedium(len(class_names)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=7, factor=0.5)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
patience_counter = 0

# --- TRENING LOOP ---
print(f"Trening započet na: {DEVICE}")
for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}", leave=False)
    for bx, by in pbar:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * bx.size(0)
        t_corr += (out.argmax(1) == by).sum().item()
        t_total += bx.size(0)
    
    model.eval()
    v_loss, v_corr, v_total = 0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            out = model(bx)
            loss = criterion(out, by)
            v_loss += loss.item() * bx.size(0)
            v_corr += (out.argmax(1) == by).sum().item()
            v_total += bx.size(0)

    # Statistika
    history['train_loss'].append(t_loss/t_total)
    history['val_loss'].append(v_loss/v_total)
    history['train_acc'].append(100*t_corr/t_total)
    history['val_acc'].append(100*v_corr/v_total)
    
    scheduler.step(v_loss/v_total)
    print(f"Epoch {epoch+1:03d} | Train Acc: {history['train_acc'][-1]:.1f}% | Train Loss: {history['train_loss'][-1]:.3f} |Val Acc: {history['val_acc'][-1]:.1f}% | Val Loss: {history['val_loss'][-1]:.3f}")

    # Best Model Save & Early Stopping
    if (v_loss/v_total) < best_val_loss:
        best_val_loss = v_loss/v_total
        torch.save(model.state_dict(), os.path.join(RUN_DIR, "best_model.pth"))
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\n[!] Early stopping na epohi {epoch+1}")
        break

# --- FINALNI REPORT ---
def save_reports():
    print("\nGeneriram izvještaje...")
    # 1. Krivulje
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1); plt.plot(history['train_loss'], label='Train'); plt.plot(history['val_loss'], label='Val'); plt.title('Loss'); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(history['train_acc'], label='Train'); plt.plot(history['val_acc'], label='Val'); plt.title('Accuracy'); plt.legend()
    plt.savefig(os.path.join(RUN_DIR, "training_curves.png"))

    # 2. Sanity check spektrograma (FIX: .squeeze())
    sample_spec, _ = next(iter(val_loader))
    plt.figure(figsize=(6, 4))
    plt.imshow(sample_spec[0].squeeze().cpu().numpy(), aspect='auto', origin='lower')
    plt.title("Model Input Sample")
    plt.savefig(os.path.join(RUN_DIR, "input_sample.png"))

    # 3. Test Evaluation
    model.load_state_dict(torch.load(os.path.join(RUN_DIR, "best_model.pth")))
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            out = model(bx.to(DEVICE))
            y_pred.extend(out.argmax(1).cpu().numpy())
            y_true.extend(by.numpy())

    with open(os.path.join(RUN_DIR, "report.txt"), "w") as f:
        f.write(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    # 4. Normalizirana matrica
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-7)
    plt.figure(figsize=(18, 14))
    sns.heatmap(cm_norm, annot=False, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.savefig(os.path.join(RUN_DIR, "confusion_matrix.png"))
    plt.close('all')

save_reports()
print(f"✅ Završeno. Rezultati su u: {RUN_DIR}")



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 48  # Povećano radi stabilnosti gradijenta
EPOCHS = 100     # Dajemo mu malo više vremena
LEARNING_RATE = 0.0008 # Malo sporiji start
WEIGHT_DECAY = 0.02    # Jača kazna za prevelike težine
EARLY_STOP_PATIENCE = 20

def get_run_folder(base_name="run"):
    n = 1
    while os.path.exists(f"{base_name}_{n}"): n += 1
    run_dir = f"{base_name}_{n}"
    os.makedirs(run_dir)
    return run_dir

RUN_DIR = get_run_folder()
print(f"\n🚀 POKRETANJE FINO-TUNINGA: Rezultati u {RUN_DIR}")

# --- MODEL (Dublja verzija s 5 slojeva) ---
class VoiceNetDeep(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )
        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256), # Novi dublji sloj
            conv_block(256, 256)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# --- DATASET (Balansirana Augmentacija) ---
class VoiceDataset(Dataset):
    def __init__(self, file_paths, labels, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment
        # Smanjeno maskiranje da model ne "oslijepi"
        self.freq_mask = torchaudio.transforms.FrequencyMasking(20) 
        self.time_mask = torchaudio.transforms.TimeMasking(35)

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        path, label = self.file_paths[idx], self.labels[idx]
        waveform, sr = torchaudio.load(path)
        
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(0, keepdim=True)
            
        if self.augment:
            # Time shift ostavljamo - on je zlata vrijedan
            shift = int(waveform.shape[1] * 0.1)
            shift_val = np.random.randint(-shift, shift)
            waveform = torch.roll(waveform, shift_val, dims=1)
            # Smanjen šum da ne uništi vokalne karakteristike
            waveform = waveform + 0.002 * torch.randn_like(waveform)

        mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        )(waveform)
        mel_spec = torchaudio.transforms.AmplitudeToDB()(mel_spec)
        
        if self.augment:
            mel_spec = self.freq_mask(mel_spec)
            mel_spec = self.time_mask(mel_spec)
            
        mel_spec = torch.nn.functional.interpolate(
            mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False
        ).squeeze(0)
        
        # Obavezna normalizacija
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-7)
        return mel_spec, label

# --- PRIPREMA ---
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
class_names = [os.path.basename(f) for f in person_folders]

all_files, all_labels = [], []
for label, folder in enumerate(person_folders):
    for f in glob.glob(os.path.join(folder, "**/*.wav"), recursive=True):
        all_files.append(f); all_labels.append(label)

X_train_f, X_temp_f, y_train, y_temp = train_test_split(all_files, all_labels, test_size=0.2, stratify=all_labels, random_state=42)
X_val_f, X_test_f, y_val, y_test = train_test_split(X_temp_f, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_loader = DataLoader(VoiceDataset(X_train_f, y_train, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(VoiceDataset(X_val_f, y_val, augment=False), batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(VoiceDataset(X_test_f, y_test, augment=False), batch_size=BATCH_SIZE)

# --- TRENING ---
model = VoiceNetDeep(len(class_names)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

history = {'t_loss': [], 'v_loss': [], 't_acc': [], 'v_acc': []}
best_v_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Ep {epoch+1:03d}", leave=False)
    for bx, by in pbar:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad(); out = model(bx); loss = criterion(out, by)
        loss.backward(); optimizer.step()
        t_loss += loss.item()*bx.size(0); t_corr += (out.argmax(1)==by).sum().item(); t_total += bx.size(0)
    
    model.eval()
    v_loss, v_corr, v_total = 0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            out = model(bx); loss = criterion(out, by)
            v_loss += loss.item()*bx.size(0); v_corr += (out.argmax(1)==by).sum().item(); v_total += bx.size(0)

    tl, ta = t_loss/t_total, 100*t_corr/t_total
    vl, va = v_loss/v_total, 100*v_corr/v_total
    history['t_loss'].append(tl); history['v_loss'].append(vl); history['t_acc'].append(ta); history['v_acc'].append(va)
    
    scheduler.step(vl)
    print(f"Ep {epoch+1:03d} | Train: {ta:.1f}% | Val: {va:.1f}% | Loss V: {vl:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if vl < best_v_loss:
        best_v_loss = vl
        torch.save(model.state_dict(), os.path.join(RUN_DIR, "best_model.pth"))
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= EARLY_STOP_PATIENCE:
        print("\n[!] Prekidamo - nema više napretka."); break

# --- FINALNI REPORT ---
print("\n📊 Čuvam rezultate u " + RUN_DIR)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.plot(history['t_loss'], label='T'); plt.plot(history['v_loss'], label='V'); plt.title('Loss'); plt.legend()
plt.subplot(1, 2, 2); plt.plot(history['t_acc'], label='T'); plt.plot(history['v_acc'], label='V'); plt.title('Acc'); plt.legend()
plt.savefig(os.path.join(RUN_DIR, "training_curves.png"))

model.load_state_dict(torch.load(os.path.join(RUN_DIR, "best_model.pth")))
model.eval()
y_t, y_p = [], []
with torch.no_grad():
    for bx, by in test_loader:
        out = model(bx.to(DEVICE))
        y_p.extend(out.argmax(1).cpu().numpy()); y_t.extend(by.numpy())

with open(os.path.join(RUN_DIR, "report.txt"), "w") as f:
    f.write(classification_report(y_t, y_p, target_names=class_names))

cm = confusion_matrix(y_t, y_p)
plt.figure(figsize=(16, 12))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.savefig(os.path.join(RUN_DIR, "confusion_matrix.png"))

print(f"Gotovo!!")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 48
EPOCHS = 100
LEARNING_RATE = 0.0008
WEIGHT_DECAY = 0.02
EARLY_STOP_PATIENCE = 20

def get_run_folder(base_name="run"):
    n = 1
    while os.path.exists(f"{base_name}_{n}"): n += 1
    run_dir = f"{base_name}_{n}"
    os.makedirs(run_dir)
    return run_dir

RUN_DIR = get_run_folder()
print(f"\n🚀 START: Full Diagnostic Run in {RUN_DIR}")

# --- MODEL (VoiceNetDeep) ---
class VoiceNetDeep(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )
        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256),
            conv_block(256, 256)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# --- OPTIMIZED DATASET ---
class VoiceDataset(Dataset):
    def __init__(self, file_paths, labels, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment
        
        # Initialize transforms once for speed
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(20) 
        self.time_mask = torchaudio.transforms.TimeMasking(35)

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        path, label = self.file_paths[idx], self.labels[idx]
        waveform, sr = torchaudio.load(path)
        
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(0, keepdim=True)
            
        if self.augment:
            shift = int(waveform.shape[1] * 0.1)
            shift_val = np.random.randint(-shift, shift)
            waveform = torch.roll(waveform, shift_val, dims=1)
            waveform = waveform + 0.002 * torch.randn_like(waveform)

        mel_spec = self.mel_transform(waveform)
        mel_spec = self.db_transform(mel_spec)
        
        if self.augment:
            mel_spec = self.freq_mask(mel_spec)
            mel_spec = self.time_mask(mel_spec)
            
        mel_spec = torch.nn.functional.interpolate(
            mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False
        ).squeeze(0)
        
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-7)
        return mel_spec, label

# --- DATA PREPARATION ---
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
class_names = [os.path.basename(f) for f in person_folders]

all_files, all_labels = [], []
for label, folder in enumerate(person_folders):
    for f in glob.glob(os.path.join(folder, "**/*.wav"), recursive=True):
        all_files.append(f); all_labels.append(label)

# Calculate Class Weights
weights = compute_class_weight(class_weight='balanced', classes=np.unique(all_labels), y=all_labels)
class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)

X_train_f, X_temp_f, y_train, y_temp = train_test_split(all_files, all_labels, test_size=0.2, stratify=all_labels, random_state=42)
X_val_f, X_test_f, y_val, y_test = train_test_split(X_temp_f, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_loader = DataLoader(VoiceDataset(X_train_f, y_train, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(VoiceDataset(X_val_f, y_val, augment=False), batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(VoiceDataset(X_test_f, y_test, augment=False), batch_size=BATCH_SIZE)

# --- TRAINING SETUP ---
model = VoiceNetDeep(len(class_names)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1, weight=class_weights)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

history = {'t_loss': [], 'v_loss': [], 't_acc': [], 'v_acc': []}
best_v_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}", leave=False)
    for bx, by in pbar:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad(); out = model(bx); loss = criterion(out, by)
        loss.backward(); optimizer.step()
        t_loss += loss.item()*bx.size(0); t_corr += (out.argmax(1)==by).sum().item(); t_total += bx.size(0)
    
    model.eval()
    v_loss, v_corr, v_total = 0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            out = model(bx); loss = criterion(out, by)
            v_loss += loss.item()*bx.size(0); v_corr += (out.argmax(1)==by).sum().item(); v_total += bx.size(0)

    tl, ta = t_loss/t_total, 100*t_corr/t_total
    vl, va = v_loss/v_total, 100*v_corr/v_total
    history['t_loss'].append(tl); history['v_loss'].append(vl); history['t_acc'].append(ta); history['v_acc'].append(va)
    
    scheduler.step(vl)
    print(f"Ep {epoch+1:03d} | Train Acc: {ta:.1f}% | Val Acc: {va:.1f}% | Val Loss: {vl:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if vl < best_v_loss:
        best_v_loss = vl
        torch.save(model.state_dict(), os.path.join(RUN_DIR, "best_model.pth"))
        patience_counter = 0
    else:
        patience_counter += 1
    if patience_counter >= EARLY_STOP_PATIENCE:
        print("\n[!] Early stopping triggered.")
        break

# --- DIAGNOSTICS & RESULTS ---
print(f"\n📊 Generating reports in {RUN_DIR}...")

# 1. Training Curves
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.plot(history['t_loss'], label='Train Loss', color='#1f77b4', lw=2)
plt.plot(history['v_loss'], label='Val Loss', color='#ff7f0e', lw=2)
plt.title('Training and Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['t_acc'], label='Train Acc', color='#2ca02c', lw=2)
plt.plot(history['v_acc'], label='Val Acc', color='#d62728', lw=2)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy %'); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "training_metrics.png"))

# 2. Confusion Matrix & Classification Report
model.load_state_dict(torch.load(os.path.join(RUN_DIR, "best_model.pth")))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for bx, by in test_loader:
        out = model(bx.to(DEVICE))
        y_pred.extend(out.argmax(1).cpu().numpy())
        y_true.extend(by.numpy())

with open(os.path.join(RUN_DIR, "detailed_report.txt"), "w") as f:
    f.write("--- CLASSIFICATION REPORT ---\n")
    f.write(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Speaker Classification')
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.savefig(os.path.join(RUN_DIR, "confusion_matrix.png"))

print(f"✅ FINISHED. Check the {RUN_DIR} folder for your results.")

In [ ]:
# kinda shit
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32  # Slightly smaller for better generalization
EPOCHS = 120     # More epochs for potential grokking
MAX_LR = 1e-3
WEIGHT_DECAY = 0.05 # Higher weight decay for better regularization

def get_run_folder(base_name="run"):
    n = 1
    while os.path.exists(f"{base_name}_{n}"): n += 1
    run_dir = f"{base_name}_{n}"
    os.makedirs(run_dir)
    return run_dir

RUN_DIR = get_run_folder()
print(f"\n🚀 START: Residual VoiceNet Training in {RUN_DIR}")

# --- MODEL COMPONENTS ---
class SEBlock(nn.Module):
    """Squeeze-and-Excitation block for channel-wise attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResBlock(nn.Module):
    def __init__(self, in_f, out_f, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_f, out_f, 3, stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_f)
        self.conv2 = nn.Conv2d(out_f, out_f, 3, 1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_f)
        self.relu = nn.ReLU(inplace=True)
        self.se = SEBlock(out_f)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_f != out_f:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_f, out_f, 1, stride, bias=False),
                nn.BatchNorm2d(out_f)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += self.shortcut(x)
        return self.relu(out)

class VoiceResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.in_planes = 32
        self.prep = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.layer1 = self._make_layer(64, 2)
        self.layer2 = self._make_layer(128, 2)
        self.layer3 = self._make_layer(256, 2)
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def _make_layer(self, planes, num_blocks):
        strides = [2] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(ResBlock(self.in_planes, planes, stride))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.classifier(x)

# --- DATASET ---
class VoiceDataset(Dataset):
    def __init__(self, file_paths, labels, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(30) # Increased mask
        self.time_mask = torchaudio.transforms.TimeMasking(40) # Increased mask

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        path, label = self.file_paths[idx], self.labels[idx]
        waveform, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(0, keepdim=True)
            
        if self.augment:
            shift = int(waveform.shape[1] * 0.15) # Increased shift
            shift_val = np.random.randint(-shift, shift)
            waveform = torch.roll(waveform, shift_val, dims=1)
            waveform = waveform + 0.003 * torch.randn_like(waveform)

        mel_spec = self.mel_transform(waveform)
        mel_spec = self.db_transform(mel_spec)
        
        if self.augment:
            mel_spec = self.freq_mask(mel_spec)
            mel_spec = self.time_mask(mel_spec)
            
        mel_spec = torch.nn.functional.interpolate(
            mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False
        ).squeeze(0)
        
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-7)
        return mel_spec, label

# --- PREPARATION ---
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
class_names = [os.path.basename(f) for f in person_folders]
all_files, all_labels = [], []
for label, folder in enumerate(person_folders):
    for f in glob.glob(os.path.join(folder, "**/*.wav"), recursive=True):
        all_files.append(f); all_labels.append(label)

weights = compute_class_weight(class_weight='balanced', classes=np.unique(all_labels), y=all_labels)
class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)

X_train_f, X_temp_f, y_train, y_temp = train_test_split(all_files, all_labels, test_size=0.2, stratify=all_labels, random_state=42)
X_val_f, X_test_f, y_val, y_test = train_test_split(X_temp_f, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_loader = DataLoader(VoiceDataset(X_train_f, y_train, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(VoiceDataset(X_val_f, y_val, augment=False), batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(VoiceDataset(X_test_f, y_test, augment=False), batch_size=BATCH_SIZE)

# --- TRAINING SETUP ---
model = VoiceResNet(len(class_names)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=MAX_LR/10, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1, weight=class_weights)

# OneCycleLR helps the model "grok" by finding better minima
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=MAX_LR, 
                                          steps_per_epoch=len(train_loader), epochs=EPOCHS)

history = {'t_loss': [], 'v_loss': [], 't_acc': [], 'v_acc': []}
best_v_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}", leave=False)
    for bx, by in pbar:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad(); out = model(bx); loss = criterion(out, by)
        loss.backward(); optimizer.step(); scheduler.step()
        t_loss += loss.item()*bx.size(0); t_corr += (out.argmax(1)==by).sum().item(); t_total += bx.size(0)
    
    model.eval()
    v_loss, v_corr, v_total = 0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            out = model(bx); loss = criterion(out, by)
            v_loss += loss.item()*bx.size(0); v_corr += (out.argmax(1)==by).sum().item(); v_total += bx.size(0)

    tl, ta = t_loss/t_total, 100*t_corr/t_total
    vl, va = v_loss/v_total, 100*v_corr/v_total
    history['t_loss'].append(tl); history['v_loss'].append(vl); history['t_acc'].append(ta); history['v_acc'].append(va)
    
    print(f"Ep {epoch+1:03d} | Train Acc: {ta:.1f}% | Val Acc: {va:.1f}% | Val Loss: {vl:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if vl < best_v_loss:
        best_v_loss = vl
        torch.save(model.state_dict(), os.path.join(RUN_DIR, "best_model.pth"))

# --- DIAGNOSTICS & RESULTS ---
print(f"\n📊 Generating reports in {RUN_DIR}...")
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.plot(history['t_loss'], label='Train Loss'); plt.plot(history['v_loss'], label='Val Loss')
plt.title('Loss Curves'); plt.xlabel('Epoch'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['t_acc'], label='Train Acc'); plt.plot(history['v_acc'], label='Val Acc')
plt.title('Accuracy Curves'); plt.xlabel('Epoch'); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "training_metrics.png"))

model.load_state_dict(torch.load(os.path.join(RUN_DIR, "best_model.pth")))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for bx, by in test_loader:
        out = model(bx.to(DEVICE))
        y_pred.extend(out.argmax(1).cpu().numpy()); y_true.extend(by.numpy())

with open(os.path.join(RUN_DIR, "detailed_report.txt"), "w") as f:
    f.write(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Final Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('True')
plt.savefig(os.path.join(RUN_DIR, "confusion_matrix.png"))

print(f"✅ FINISHED. The Residual Model is ready.")




In [ ]:
# NAS experiments => we have a new best locally
# isprobana optuna; but fuck that
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import numpy as np
import optuna
import logging
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import sys

# --- POSTAVKE LOGIRANJA ---
logging.getLogger("optuna").setLevel(logging.WARNING)

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 48
N_TRIALS = 30 
MAX_EPOCHS_PER_TRIAL = 15

# --- DATASET ---
class VoiceDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        path, label = self.file_paths[idx], self.labels[idx]
        try:
            waveform, sr = torchaudio.load(path)
            if sr != SAMPLE_RATE:
                waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
            if waveform.shape[0] > 1:
                waveform = waveform.mean(0, keepdim=True)
            
            mel_spec = self.mel_transform(waveform)
            mel_spec = self.db_transform(mel_spec)
            mel_spec = torch.nn.functional.interpolate(
                mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False
            ).squeeze(0)
            mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-7)
            return mel_spec, label
        except Exception:
            return torch.zeros((1, 128, 128)), label

# --- DYNAMIC MODEL ---
class DynamicVoiceNet(nn.Module):
    def __init__(self, trial, num_classes):
        super().__init__()
        layers = []
        n_layers = trial.suggest_int("n_layers", 3, 6)
        in_channels = 1
        for i in range(n_layers):
            out_channels = trial.suggest_int(f"units_l{i}", 16, 128, step=16)
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2))
            in_channels = out_channels
        
        self.features = nn.Sequential(*layers)
        dropout_rate = trial.suggest_float("dropout", 0.2, 0.5)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(in_channels, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# --- NAS OBJECTIVE ---
def objective(trial):
    current_best = "N/A"
    try:
        current_best = f"{trial.study.best_value:.4f}"
    except Exception: pass

    print(f"\n\033[94m{'━'*70}\033[0m")
    print(f"🚀 \033[1mTRIALS #{trial.number}/{N_TRIALS}\033[0m | \033[92mBEST SO FAR: {current_best}\033[0m")
    
    person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
    class_names = [os.path.basename(f) for f in person_folders]
    all_files, all_labels = [], []
    for label, folder in enumerate(person_folders):
        for f in glob.glob(os.path.join(folder, "**/*.wav"), recursive=True):
            all_files.append(f); all_labels.append(label)

    X_train, X_val, y_train, y_val = train_test_split(all_files, all_labels, test_size=0.2, stratify=all_labels)
    train_loader = DataLoader(VoiceDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(VoiceDataset(X_val, y_val), batch_size=BATCH_SIZE)

    model = DynamicVoiceNet(trial, len(class_names)).to(DEVICE)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    print(f"⚙️  \033[2mLR: {lr:.6f} | Layers: {trial.params['n_layers']} | Dropout: {trial.params['dropout']:.2f}\033[0m")

    for epoch in range(MAX_EPOCHS_PER_TRIAL):
        # --- TRENING FAZA ---
        model.train()
        train_loss, t_corr, t_total = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"   Ep {epoch+1:02d}/{MAX_EPOCHS_PER_TRIAL}", leave=False, bar_format='{l_bar}{bar:20}{r_bar}')
        
        for bx, by in pbar:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            t_corr += (out.argmax(1) == by).sum().item()
            t_total += bx.size(0)
            pbar.set_postfix(loss=f"{loss.item():.3f}")

        # --- VALIDACIJSKA FAZA ---
        model.eval()
        val_loss, v_corr, v_total = 0.0, 0, 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                out = model(bx)
                v_loss = criterion(out, by)
                val_loss += v_loss.item()
                v_corr += (out.argmax(1) == by).sum().item()
                v_total += bx.size(0)
        
        # Izračun finalnih metrika za epohu
        avg_t_loss = train_loss / len(train_loader)
        avg_v_loss = val_loss / len(val_loader)
        train_acc = t_corr / t_total
        val_acc = v_corr / v_total

        # Verbose ispis sa svim traženim metrikama
        print(f"   📊 Ep {epoch+1:02d} -> "
              f"T-Loss: \033[90m{avg_t_loss:.4f}\033[0m | T-Acc: \033[90m{train_acc:.4f}\033[0m | "
              f"V-Loss: \033[95m{avg_v_loss:.4f}\033[0m | V-Acc: \033[1m\033[92m{val_acc:.4f}\033[0m")

        # Optuna Pruning
        trial.report(val_acc, epoch)
        if trial.should_prune():
            print(f"   \033[91m❌ PRUNED\033[0m at epoch {epoch+1}")
            raise optuna.exceptions.TrialPruned()

    return val_acc

if __name__ == "__main__":
    print("\n\033[1m" + "="*70)
    print("  VOICE RECOGNITION - ULTIMATE VERBOSE NAS")
    print("="*70 + "\033[0m")
    
    study = optuna.create_study(
        direction="maximize", 
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=3)
    )
    
    try:
        study.optimize(objective, n_trials=N_TRIALS)
    except KeyboardInterrupt:
        print("\n\033[93m⚠️ Prekinuto. Analiziram najbolje rezultate...\033[0m")

    if len(study.trials) > 0:
        print("\n" + "\033[1m\033[92m" + "═"*70)
        print("🏆 POBJEDNIČKA ARHITEKTURA 🏆")
        print(f"Finalna Točnost: {study.best_value:.4f}")
        print("\nOptimalni parametri:")
        for key, value in study.best_params.items():
            print(f" 🔹 {key}: {value}")
        print("═"*70 + "\033[0m")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32 # Smanjeno zbog kompleksnije arhitekture
EPOCHS = 60
MAX_LR = 1e-3
WEIGHT_DECAY = 0.05

def get_run_folder(base_name="run"):
    n = 1
    while os.path.exists(f"{base_name}_{n}"): n += 1
    run_dir = f"{base_name}_{n}"
    os.makedirs(run_dir)
    return run_dir

RUN_DIR = get_run_folder()
print(f"\n🔥 START: ResNet-SE Diagnostic Run in {RUN_DIR}")

# --- MODEL COMPONENTS (ResNet + Attention) ---
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = F.adaptive_avg_pool2d(x, 1).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class ResBlock(nn.Module):
    def __init__(self, in_f, out_f, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_f, out_f, 3, stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_f)
        self.conv2 = nn.Conv2d(out_f, out_f, 3, 1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_f)
        self.se = SEBlock(out_f)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_f != out_f:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_f, out_f, 1, stride, bias=False),
                nn.BatchNorm2d(out_f)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += self.shortcut(x)
        return F.relu(out)

class VoiceResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        
        # 4 faze s po 2 bloka (ukupno 8 konvolucijskih blokova)
        self.layer1 = self._make_layer(32, 32, stride=1)
        self.layer2 = self._make_layer(32, 64, stride=2)
        self.layer3 = self._make_layer(64, 128, stride=2)
        self.layer4 = self._make_layer(128, 256, stride=2)
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def _make_layer(self, in_f, out_f, stride):
        layers = [ResBlock(in_f, out_f, stride), ResBlock(out_f, out_f, 1)]
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.classifier(x)

# --- DATASET ---
class VoiceDataset(Dataset):
    def __init__(self, file_paths, labels, augment=False):
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(15) 
        self.time_mask = torchaudio.transforms.TimeMasking(30)

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        path, label = self.file_paths[idx], self.labels[idx]
        try:
            waveform, sr = torchaudio.load(path)
            if sr != SAMPLE_RATE:
                waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
            if waveform.shape[0] > 1: waveform = waveform.mean(0, keepdim=True)
            
            if self.augment:
                waveform = waveform + 0.003 * torch.randn_like(waveform)

            mel_spec = self.db_transform(self.mel_transform(waveform))
            
            if self.augment:
                mel_spec = self.freq_mask(mel_spec)
                mel_spec = self.time_mask(mel_spec)

            mel_spec = F.interpolate(mel_spec.unsqueeze(0), size=(128, 128), mode='bilinear').squeeze(0)
            mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-7)
            return mel_spec, label
        except:
            return torch.zeros((1, 128, 128)), label

# --- PREP DATA ---
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
class_names = [os.path.basename(f) for f in person_folders]
all_files, all_labels = [], []
for label, folder in enumerate(person_folders):
    for f in glob.glob(os.path.join(folder, "**/*.wav"), recursive=True):
        all_files.append(f); all_labels.append(label)

weights = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)

X_train, X_temp, y_train, y_temp = train_test_split(all_files, all_labels, test_size=0.2, stratify=all_labels, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_loader = DataLoader(VoiceDataset(X_train, y_train, augment=True), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(VoiceDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(VoiceDataset(X_test, y_test), batch_size=BATCH_SIZE)

# --- SETUP ---
model = VoiceResNet(len(class_names)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1, weight=class_weights)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=MAX_LR, steps_per_epoch=len(train_loader), epochs=EPOCHS)

history = {'t_loss': [], 'v_loss': [], 't_acc': [], 'v_acc': []}
best_acc = 0

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_corr, t_total = 0, 0, 0
    for bx, by in tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False):
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        scheduler.step() # Unutar batch-a!
        
        t_loss += loss.item() * bx.size(0)
        t_corr += (out.argmax(1) == by).sum().item()
        t_total += bx.size(0)

    model.eval()
    v_loss, v_corr, v_total = 0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            out = model(bx)
            v_loss += criterion(out, by).item() * bx.size(0)
            v_corr += (out.argmax(1) == by).sum().item()
            v_total += bx.size(0)

    history['t_loss'].append(t_loss/t_total); history['v_loss'].append(v_loss/v_total)
    history['t_acc'].append(100*t_corr/t_total); history['v_acc'].append(100*v_corr/v_total)
    
    print(f"Ep {epoch+1:03d} | T-Acc: {history['t_acc'][-1]:.1f}% | V-Acc: {history['v_acc'][-1]:.1f}% | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if history['v_acc'][-1] > best_acc:
        best_acc = history['v_acc'][-1]
        torch.save({'model': model.state_dict(), 'classes': class_names}, os.path.join(RUN_DIR, "best_model.pth"))

# --- DIAGNOSTICS ---
model.load_state_dict(torch.load(os.path.join(RUN_DIR, "best_model.pth"))['model'])
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for bx, by in test_loader:
        out = model(bx.to(DEVICE))
        y_pred.extend(out.argmax(1).cpu().numpy()); y_true.extend(by.numpy())

plt.figure(figsize=(15,5))
plt.subplot(1,2,1); plt.plot(history['t_loss'], label='Train'); plt.plot(history['v_loss'], label='Val'); plt.legend(); plt.title("Loss")
plt.subplot(1,2,2); plt.plot(history['t_acc'], label='Train'); plt.plot(history['v_acc'], label='Val'); plt.legend(); plt.title("Acc %")
plt.savefig(os.path.join(RUN_DIR, "curves.png"))

plt.figure(figsize=(12,10))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.savefig(os.path.join(RUN_DIR, "cm.png"))

print(f"✅ GOTOVO. Rezultati u {RUN_DIR}")

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
PRETRAINED_PATH = "run_13/best_model.pth" 
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32 
EPOCHS = 100 # Povećano jer imamo Early Stopping
LEARNING_RATE = 0.00005 
EMBEDDING_DIM = 128
MARGIN = 1.0

# Early Stopping pragovi
TARGET_ACCURACY = 99.90
TARGET_LOSS = 0.10

def get_run_folder(base_name="triplet_run"):
    n = 1
    while os.path.exists(f"{base_name}_{n}"): n += 1
    run_dir = f"{base_name}_{n}"
    os.makedirs(run_dir)
    return run_dir

RUN_DIR = get_run_folder()

# --- MODEL (VoiceNetEmbedding) ---
class VoiceNetEmbedding(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )
        self.features = nn.Sequential(
            conv_block(1, 32),   
            conv_block(32, 64),  
            conv_block(64, 128), 
            conv_block(128, 256),
            conv_block(256, 256) 
        )
        self.embedding_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, embedding_dim)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.embedding_head(x)
        return torch.nn.functional.normalize(x, p=2, dim=1)

# --- TRIPLET DATASET ---
class TripletVoiceDataset(Dataset):
    def __init__(self, folder_map, is_val=False):
        self.folder_map = folder_map
        self.class_ids = list(folder_map.keys())
        self.all_files = []
        for cid in self.class_ids:
            for f in folder_map[cid]:
                self.all_files.append((f, cid))
        
        self.is_val = is_val
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.all_files)

    def _get_spec(self, path, apply_aug=False):
        waveform, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1: waveform = waveform.mean(0, keepdim=True)
        
        if apply_aug:
            shift = int(waveform.shape[1] * 0.1)
            waveform = torch.roll(waveform, random.randint(-shift, shift), dims=1)
            waveform = waveform + 0.005 * torch.randn_like(waveform)

        spec = self.db_transform(self.mel_transform(waveform))
        spec = torch.nn.functional.interpolate(spec.unsqueeze(0), size=(128, 128)).squeeze(0)
        return (spec - spec.mean()) / (spec.std() + 1e-7)

    def __getitem__(self, idx):
        anchor_path, anchor_label = self.all_files[idx]
        
        pos_list = self.folder_map[anchor_label]
        if len(pos_list) > 1:
            pos_path = random.choice([p for p in pos_list if p != anchor_path])
            pos_spec = self._get_spec(pos_path, apply_aug=not self.is_val)
        else:
            pos_spec = self._get_spec(anchor_path, apply_aug=True)
        
        neg_label = random.choice([c for c in self.class_ids if c != anchor_label])
        neg_path = random.choice(self.folder_map[neg_label])
        
        return self._get_spec(anchor_path), pos_spec, self._get_spec(neg_path)

# --- PRIPREMA PODATAKA ---
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
folder_map = {i: glob.glob(os.path.join(f, "**/*.wav"), recursive=True) for i, f in enumerate(person_folders)}
folder_map = {k: v for k, v in folder_map.items() if len(v) > 0}

train_loader = DataLoader(TripletVoiceDataset(folder_map), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TripletVoiceDataset(folder_map, is_val=True), batch_size=BATCH_SIZE)

# --- INICIJALIZACIJA ---
model = VoiceNetEmbedding(EMBEDDING_DIM).to(DEVICE)

if os.path.exists(PRETRAINED_PATH):
    print(f"💉 Učitavam bazu znanja iz {PRETRAINED_PATH}...")
    old_state = torch.load(PRETRAINED_PATH, map_location=DEVICE)
    model_dict = model.state_dict()
    pretrained_features = {k: v for k, v in old_state.items() if k.startswith('features')}
    model_dict.update(pretrained_features)
    model.load_state_dict(model_dict)
    print("✅ Konvolucijski slojevi uspješno presuđeni.")

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
criterion = nn.TripletMarginLoss(margin=MARGIN, p=2)

# --- TRENING PETLJA ---
history = {'epoch': [], 'loss': [], 'val_triplet_acc': []}

print(f"🚀 Krećem s treningom. Cilj: Loss < {TARGET_LOSS} & Acc > {TARGET_ACCURACY}%")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS} [Train]")
    
    for anchor, pos, neg in pbar:
        anchor, pos, neg = anchor.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
        
        optimizer.zero_grad()
        a_emb, p_emb, n_emb = model(anchor), model(pos), model(neg)
        
        loss = criterion(a_emb, p_emb, n_emb)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})

    avg_loss = epoch_loss / len(train_loader)

    # --- EVALUACIJA ---
    model.eval()
    correct_triplets = 0
    total_triplets = 0
    with torch.no_grad():
        for anchor, pos, neg in val_loader:
            anchor, pos, neg = anchor.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
            a_emb, p_emb, n_emb = model(anchor), model(pos), model(neg)
            
            dist_pos = (a_emb - p_emb).pow(2).sum(1)
            dist_neg = (a_emb - n_emb).pow(2).sum(1)
            
            correct_triplets += (dist_pos < dist_neg).sum().item()
            total_triplets += anchor.size(0)
    
    t_acc = 100 * correct_triplets / total_triplets
    
    # Logiranje
    history['epoch'].append(epoch + 1)
    history['loss'].append(avg_loss)
    history['val_triplet_acc'].append(t_acc)
    
    print(f"✨ Ep {epoch+1} | Loss: {avg_loss:.4f} | Triplet Accuracy: {t_acc:.2f}%")
    
    # --- EARLY STOPPING PROVJERA ---
    if t_acc >= TARGET_ACCURACY and avg_loss <= TARGET_LOSS:
        print(f"\n🎯 META POGOĐENA! (Acc: {t_acc:.2f}%, Loss: {avg_loss:.4f})")
        torch.save(model.state_dict(), os.path.join(RUN_DIR, "best_triplet_model.pth"))
        break
    
    # Backup svakih 10 epoha
    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(), os.path.join(RUN_DIR, f"checkpoint_ep{epoch+1}.pth"))

# --- SPREMANJE I VIZUALIZACIJA ---
pd.DataFrame(history).to_csv(os.path.join(RUN_DIR, "log.csv"), index=False)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['loss'], label='Triplet Loss', color='red')
plt.axhline(y=TARGET_LOSS, color='gray', linestyle='--', label='Target Loss')
plt.title('Gubitak kroz epohe'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['val_triplet_acc'], label='Triplet Accuracy', color='green')
plt.axhline(y=TARGET_ACCURACY, color='gray', linestyle='--', label='Target Acc')
plt.title('Točnost razdvajanja (%)'); plt.legend()

plt.savefig(os.path.join(RUN_DIR, "metrics.png"))
plt.show()

print(f"✅ Trening završen. Sve je u folderu: {RUN_DIR}")

💉 Učitavam bazu znanja iz run_13/best_model.pth...
✅ Konvolucijski slojevi uspješno presuđeni.
🚀 Krećem s treningom. Cilj: Loss < 0.1 & Acc > 99.9%


Ep 1/100 [Train]:   3%|▎         | 4/153 [00:27<17:20,  6.99s/it, loss=0.793]


KeyboardInterrupt: 

In [ ]:
"""
VoiceNet Embedding Extractor: Triplet Learning Framework

This script transforms a standard convolutional neural network (CNN) into a 
discriminative feature extractor (embedding model) for voice verification. 

Key architectural and procedural shifts implemented:

1. Architecture Adaptation: The network's classification head (Softmax layer) 
   is replaced with a linear 'Embedding Head' and an L2 normalization layer. 
   Instead of predicting discrete classes, the model maps voice Mel-spectrograms 
   into a 128-dimensional hypersphere where semantic similarity is measured 
   by Euclidean distance.

2. Triplet Loss Paradigm: We shift from Cross-Entropy to Triplet Margin Loss. 
   During training, the model processes triplets: an 'Anchor' (sample A), 
   a 'Positive' (another sample of the same speaker), and a 'Negative' 
   (a sample from a different speaker). 

3. Optimization Objective: The model is trained to minimize the distance 
   between Anchor-Positive pairs while maximizing the distance between 
   Anchor-Negative pairs by at least a predefined 'Margin' (1.0).

4. Feature Transfer: The script supports partial weight loading, allowing 
   the model to inherit spatial feature extraction capabilities from a 
   pretrained classifier ('features' blocks) while fine-tuning the 
   embedding space for verification tasks.

The result is a model that can compare two previously unseen voices and 
determine if they belong to the same person based on their spatial proximity 
in the embedding manifold.
"""


import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import os
import glob
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
PRETRAINED_PATH = "run_13/best_model.pth" 
SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32 
EPOCHS = 100 # Povećano jer imamo Early Stopping
LEARNING_RATE = 0.00005 
EMBEDDING_DIM = 128
MARGIN = 1.0

# Early Stopping pragovi
TARGET_ACCURACY = 99.90
TARGET_LOSS = 0.10

def get_run_folder(base_name="triplet_run"):
    n = 1
    while os.path.exists(f"{base_name}_{n}"): n += 1
    run_dir = f"{base_name}_{n}"
    os.makedirs(run_dir)
    return run_dir

RUN_DIR = get_run_folder()

# --- MODEL (VoiceNetEmbedding) ---
class VoiceNetEmbedding(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )
        self.features = nn.Sequential(
            conv_block(1, 32),   
            conv_block(32, 64),  
            conv_block(64, 128), 
            conv_block(128, 256),
            conv_block(256, 256) 
        )
        self.embedding_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, embedding_dim)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.embedding_head(x)
        return torch.nn.functional.normalize(x, p=2, dim=1)

# --- TRIPLET DATASET ---
class TripletVoiceDataset(Dataset):
    def __init__(self, folder_map, is_val=False):
        self.folder_map = folder_map
        self.class_ids = list(folder_map.keys())
        self.all_files = []
        for cid in self.class_ids:
            for f in folder_map[cid]:
                self.all_files.append((f, cid))
        
        self.is_val = is_val
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.all_files)

    def _get_spec(self, path, apply_aug=False):
        waveform, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1: waveform = waveform.mean(0, keepdim=True)
        
        if apply_aug:
            shift = int(waveform.shape[1] * 0.1)
            waveform = torch.roll(waveform, random.randint(-shift, shift), dims=1)
            waveform = waveform + 0.005 * torch.randn_like(waveform)

        spec = self.db_transform(self.mel_transform(waveform))
        spec = torch.nn.functional.interpolate(spec.unsqueeze(0), size=(128, 128)).squeeze(0)
        return (spec - spec.mean()) / (spec.std() + 1e-7)

    def __getitem__(self, idx):
        anchor_path, anchor_label = self.all_files[idx]
        
        pos_list = self.folder_map[anchor_label]
        if len(pos_list) > 1:
            pos_path = random.choice([p for p in pos_list if p != anchor_path])
            pos_spec = self._get_spec(pos_path, apply_aug=not self.is_val)
        else:
            pos_spec = self._get_spec(anchor_path, apply_aug=True)
        
        neg_label = random.choice([c for c in self.class_ids if c != anchor_label])
        neg_path = random.choice(self.folder_map[neg_label])
        
        return self._get_spec(anchor_path), pos_spec, self._get_spec(neg_path)

# --- PRIPREMA PODATAKA ---
person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
folder_map = {i: glob.glob(os.path.join(f, "**/*.wav"), recursive=True) for i, f in enumerate(person_folders)}
folder_map = {k: v for k, v in folder_map.items() if len(v) > 0}

train_loader = DataLoader(TripletVoiceDataset(folder_map), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TripletVoiceDataset(folder_map, is_val=True), batch_size=BATCH_SIZE)

# --- INICIJALIZACIJA ---
model = VoiceNetEmbedding(EMBEDDING_DIM).to(DEVICE)

if os.path.exists(PRETRAINED_PATH):
    print(f"💉 Učitavam bazu znanja iz {PRETRAINED_PATH}...")
    old_state = torch.load(PRETRAINED_PATH, map_location=DEVICE)
    model_dict = model.state_dict()
    pretrained_features = {k: v for k, v in old_state.items() if k.startswith('features')}
    model_dict.update(pretrained_features)
    model.load_state_dict(model_dict)
    print("✅ Konvolucijski slojevi uspješno presuđeni.")

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
criterion = nn.TripletMarginLoss(margin=MARGIN, p=2)

# --- TRENING PETLJA ---
history = {'epoch': [], 'loss': [], 'val_triplet_acc': []}

print(f"🚀 Krećem s treningom. Cilj: Loss < {TARGET_LOSS} & Acc > {TARGET_ACCURACY}%")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS} [Train]")
    
    for anchor, pos, neg in pbar:
        anchor, pos, neg = anchor.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
        
        optimizer.zero_grad()
        a_emb, p_emb, n_emb = model(anchor), model(pos), model(neg)
        
        loss = criterion(a_emb, p_emb, n_emb)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})

    avg_loss = epoch_loss / len(train_loader)

    # --- EVALUACIJA ---
    model.eval()
    correct_triplets = 0
    total_triplets = 0
    with torch.no_grad():
        for anchor, pos, neg in val_loader:
            anchor, pos, neg = anchor.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
            a_emb, p_emb, n_emb = model(anchor), model(pos), model(neg)
            
            dist_pos = (a_emb - p_emb).pow(2).sum(1)
            dist_neg = (a_emb - n_emb).pow(2).sum(1)
            
            correct_triplets += (dist_pos < dist_neg).sum().item()
            total_triplets += anchor.size(0)
    
    t_acc = 100 * correct_triplets / total_triplets
    
    # Logiranje
    history['epoch'].append(epoch + 1)
    history['loss'].append(avg_loss)
    history['val_triplet_acc'].append(t_acc)
    
    print(f"✨ Ep {epoch+1} | Loss: {avg_loss:.4f} | Triplet Accuracy: {t_acc:.2f}%")
    
    # --- EARLY STOPPING PROVJERA ---
    if t_acc >= TARGET_ACCURACY and avg_loss <= TARGET_LOSS:
        print(f"\n🎯 META POGOĐENA! (Acc: {t_acc:.2f}%, Loss: {avg_loss:.4f})")
        torch.save(model.state_dict(), os.path.join(RUN_DIR, "best_triplet_model.pth"))
        break
    
    # Backup svakih 10 epoha
    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(), os.path.join(RUN_DIR, f"checkpoint_ep{epoch+1}.pth"))

# --- SPREMANJE I VIZUALIZACIJA ---
pd.DataFrame(history).to_csv(os.path.join(RUN_DIR, "log.csv"), index=False)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['loss'], label='Triplet Loss', color='red')
plt.axhline(y=TARGET_LOSS, color='gray', linestyle='--', label='Target Loss')
plt.title('Gubitak kroz epohe'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['val_triplet_acc'], label='Triplet Accuracy', color='green')
plt.axhline(y=TARGET_ACCURACY, color='gray', linestyle='--', label='Target Acc')
plt.title('Točnost razdvajanja (%)'); plt.legend()

plt.savefig(os.path.join(RUN_DIR, "metrics.png"))
plt.show()

print(f"✅ Trening završen. Sve je u folderu: {RUN_DIR}")

In [ ]:
import torch
import torchaudio
import torch.nn.functional as F
import os

# --- CONFIG ---
MODEL_PATH = "triplet_run_2/best_triplet_model.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 16000
EMBEDDING_DIM = 128

class VoiceNetEmbedding(torch.nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        def conv_block(in_f, out_f):
            return torch.nn.Sequential(
                torch.nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                torch.nn.BatchNorm2d(out_f),
                torch.nn.ReLU(),
                torch.nn.MaxPool2d(2)
            )
        self.features = torch.nn.Sequential(
            conv_block(1, 32), conv_block(32, 64), conv_block(64, 128),
            conv_block(128, 256), conv_block(256, 256)
        )
        self.embedding_head = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten(),
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, embedding_dim)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.embedding_head(x)
        return F.normalize(x, p=2, dim=1)

def load_audio(path):
    waveform, sr = torchaudio.load(path)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
    if waveform.shape[0] > 1: waveform = waveform.mean(0, keepdim=True)
    
    mel_tf = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512)
    db_tf = torchaudio.transforms.AmplitudeToDB()
    spec = db_tf(mel_tf(waveform))
    spec = F.interpolate(spec.unsqueeze(0), size=(128, 128)).to(DEVICE)
    spec = (spec - spec.mean()) / (spec.std() + 1e-7)
    return spec

def compare_voices(file1, file2):
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Error: Model ne postoji na putanji {MODEL_PATH}")
        return

    model = VoiceNetEmbedding(EMBEDDING_DIM).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    with torch.inference_mode():
        emb1 = model(load_audio(file1))
        emb2 = model(load_audio(file2))
        
        # Euklidska distanca
        distance = torch.pow(emb1 - emb2, 2).sum(1).sqrt().item()
        # Kosinusna sličnost
        similarity = F.cosine_similarity(emb1, emb2).item()

    print(f"\n" + "="*50)
    print(f"🔍 ANALIZA GLASA")
    print(f"="*50)
    # Ispisujemo pune putanje za maksimalnu jasnoću
    print(f"📄 PATH 1: {file1}")
    print(f"📄 PATH 2: {file2}")
    print(f"-"*50)
    print(f"📏 Euclidean distance: {distance:.4f}")
    print(f"🤝 Similarity (0-1):   {similarity:.4f}")
    print(f"-"*50)
    
    # Prag od 0.7-0.8 je obično "sweet spot" za tvoje rezultate
    if distance < 0.75: 
        print("✅ REZULTAT: ISTA OSOBA")
    else:
        print("❌ REZULTAT: RAZLIČITE OSOBE")
    print("="*50 + "\n")

# --- TESTIRANJE ---
# Možeš proslijediti apsolutne ili relativne putanje
f1 = "/home/antonio/Desktop/NMDU/dataset_voice/id10302/OMw6_X2IDvE/00001.wav"
f2 = "/home/antonio/Desktop/NMDU/dataset_voice/id10295/kTLpYHUNf5A/00002.wav"

compare_voices(f1, f2)

In [ ]:
import torch
import torchaudio
import torch.nn.functional as F
import os
import glob
import faiss
import numpy as np
import pickle
from tqdm import tqdm

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
MODEL_PATH = "triplet_run_2/best_triplet_model.pth" # Prilagodi putanju
INDEX_NAME = "voice_database.index"
METADATA_NAME = "voice_metadata.pkl"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 16000
EMBEDDING_DIM = 128

# --- MODEL (Isti kao tvoj) ---
class VoiceNetEmbedding(torch.nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        def conv_block(in_f, out_f):
            return torch.nn.Sequential(
                torch.nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                torch.nn.BatchNorm2d(out_f),
                torch.nn.ReLU(),
                torch.nn.MaxPool2d(2)
            )
        self.features = torch.nn.Sequential(
            conv_block(1, 32), conv_block(32, 64), conv_block(64, 128),
            conv_block(128, 256), conv_block(256, 256)
        )
        self.embedding_head = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten(),
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, embedding_dim)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.embedding_head(x)
        return F.normalize(x, p=2, dim=1)

def load_and_preprocess(path):
    waveform, sr = torchaudio.load(path)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
    if waveform.shape[0] > 1: waveform = waveform.mean(0, keepdim=True)
    
    mel_tf = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512)
    db_tf = torchaudio.transforms.AmplitudeToDB()
    spec = db_tf(mel_tf(waveform))
    spec = F.interpolate(spec.unsqueeze(0), size=(128, 128))
    spec = (spec - spec.mean()) / (spec.std() + 1e-7)
    return spec.to(DEVICE)

# --- BUILDING THE INDEX ---
def build_faiss_index():
    # 1. Inicijalizacija modela
    model = VoiceNetEmbedding(EMBEDDING_DIM).to(DEVICE)
    if not os.path.exists(MODEL_PATH):
        print(f"Baza znanja nije pronađena na {MODEL_PATH}!")
        return
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    # 2. Priprema FAISS indeksa
    index = faiss.IndexFlatL2(EMBEDDING_DIM)
    metadata = [] # Lista koja mapira index u bazi na info o fileu

    # 3. Iteracija kroz klase (isto kao tvoj folder_map)
    person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
    
    print(f"🚀 Krećem u izgradnju indeksa za {len(person_folders)} osoba...")

    all_embeddings = []
    
    with torch.inference_mode():
        for person_path in tqdm(person_folders, desc="Osobe"):
            person_id = os.path.basename(person_path)
            wav_files = glob.glob(os.path.join(person_path, "**/*.wav"), recursive=True)
            
            for f in wav_files:
                spec = load_and_preprocess(f)
                emb = model(spec).cpu().numpy()
                
                all_embeddings.append(emb)
                metadata.append({
                    "person_id": person_id,
                    "file_path": f
                })

    # 4. Dodavanje u FAISS (batch mode)
    if all_embeddings:
        final_embeddings = np.vstack(all_embeddings).astype('float32')
        index.add(final_embeddings)
        
        # 5. Spremanje na disk
        faiss.write_index(index, INDEX_NAME)
        with open(METADATA_NAME, 'wb') as f:
            pickle.dump(metadata, f)
            
        print(f"✅ Indeks uspješno spremljen! Broj zapisa: {index.ntotal}")
    else:
        print("Nema pronađenih podataka.")

# --- SEARCH FUNKCIJA ---
def search_voice(query_file, k=5000):
    # Učitaj resurse
    model = VoiceNetEmbedding(EMBEDDING_DIM).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    
    index = faiss.read_index(INDEX_NAME)
    with open(METADATA_NAME, 'rb') as f:
        metadata = pickle.load(f)

    # Procesiraj query
    spec = load_and_preprocess(query_file)
    with torch.inference_mode():
        query_emb = model(spec).cpu().numpy().astype('float32')

    # FAISS Search
    distances, indices = index.search(query_emb, k)

    print(f"\n🔍 Rezultati pretrage za: {os.path.basename(query_file)}")
    print("-" * 50)
    for i in range(k):
        idx = indices[0][i]
        dist = distances[0][i]
        if idx != -1:
            meta = metadata[idx]
            # S obzirom na Triplet Loss i normalizaciju, distanca < 0.6 je obično siguran match
            match_status = "✅ POGODAK" if dist < 0.6 else "❓ SLIČNO"
            print(f"{i+1}. ID: {meta['person_id']} | Dist: {dist:.4f} | {match_status}")
            print(f"   Path: {meta['file_path']}")

if __name__ == "__main__":
    # Prvo izgradi bazu
    build_faiss_index()
    
    # Primjer pretrage
    #test_file = "/home/antonio/Desktop/NMDU/dataset_voice/id10302/OMw6_X2IDvE/00001.wav"
    test_file = "/home/antonio/Desktop/NMDU/dataset_voice/id10302/VRCidrXwd1s/00013.wav"
    search_voice(test_file)

In [13]:
import torch
import torchaudio
import torch.nn.functional as F
import os

# --- CONFIG ---
MODEL_PATH = "triplet_run_2/best_triplet_model.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 16000
EMBEDDING_DIM = 128

class VoiceNetEmbedding(torch.nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        def conv_block(in_f, out_f):
            return torch.nn.Sequential(
                torch.nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                torch.nn.BatchNorm2d(out_f),
                torch.nn.ReLU(),
                torch.nn.MaxPool2d(2)
            )
        self.features = torch.nn.Sequential(
            conv_block(1, 32), conv_block(32, 64), conv_block(64, 128),
            conv_block(128, 256), conv_block(256, 256)
        )
        self.embedding_head = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten(),
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, embedding_dim)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.embedding_head(x)
        return F.normalize(x, p=2, dim=1)

def load_audio(path):
    waveform, sr = torchaudio.load(path)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
    if waveform.shape[0] > 1: waveform = waveform.mean(0, keepdim=True)
    
    mel_tf = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512)
    db_tf = torchaudio.transforms.AmplitudeToDB()
    spec = db_tf(mel_tf(waveform))
    spec = F.interpolate(spec.unsqueeze(0), size=(128, 128)).to(DEVICE)
    spec = (spec - spec.mean()) / (spec.std() + 1e-7)
    return spec

def compare_voices(file1, file2):
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Error: Model ne postoji na putanji {MODEL_PATH}")
        return

    model = VoiceNetEmbedding(EMBEDDING_DIM).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    with torch.inference_mode():
        emb1 = model(load_audio(file1))
        emb2 = model(load_audio(file2))
        
        # Euklidska distanca
        distance = torch.pow(emb1 - emb2, 2).sum(1).sqrt().item()
        # Kosinusna sličnost
        similarity = F.cosine_similarity(emb1, emb2).item()

    print(f"\n" + "="*50)
    print(f"🔍 ANALIZA GLASA")
    print(f"="*50)
    # Ispisujemo pune putanje za maksimalnu jasnoću
    print(f"📄 PATH 1: {file1}")
    print(f"📄 PATH 2: {file2}")
    print(f"-"*50)
    print(f"📏 Euclidean distance: {distance:.4f}")
    print(f"🤝 Similarity (0-1):   {similarity:.4f}")
    print(f"-"*50)
    
    # Prag od 0.7-0.8 je obično "sweet spot" za tvoje rezultate
    if distance < 0.75: 
        print("✅ REZULTAT: ISTA OSOBA")
    else:
        print("❌ REZULTAT: RAZLIČITE OSOBE")
    print("="*50 + "\n")

# --- TESTIRANJE ---
# Možeš proslijediti apsolutne ili relativne putanje
f1 = "/home/antonio/Desktop/NMDU/dataset_voice/id10302/OMw6_X2IDvE/00001.wav"
f2 = "/home/antonio/Desktop/NMDU/dataset_voice/id10295/kTLpYHUNf5A/00002.wav"

compare_voices(f1, f2)


🔍 ANALIZA GLASA
📄 PATH 1: /home/antonio/Desktop/NMDU/dataset_voice/id10302/OMw6_X2IDvE/00001.wav
📄 PATH 2: /home/antonio/Desktop/NMDU/dataset_voice/id10295/kTLpYHUNf5A/00002.wav
--------------------------------------------------
📏 Euclidean distance: 1.3765
🤝 Similarity (0-1):   0.0527
--------------------------------------------------
❌ REZULTAT: RAZLIČITE OSOBE



In [15]:
import torch
import torchaudio
import torch.nn.functional as F
import os
import glob
import faiss
import numpy as np
import pickle
from tqdm import tqdm

# --- CONFIG ---
DATASET_DIR = "dataset_voice"
MODEL_PATH = "triplet_run_2/best_triplet_model.pth" # Prilagodi putanju
INDEX_NAME = "voice_database.index"
METADATA_NAME = "voice_metadata.pkl"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 16000
EMBEDDING_DIM = 128

# --- MODEL (Isti kao tvoj) ---
class VoiceNetEmbedding(torch.nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        def conv_block(in_f, out_f):
            return torch.nn.Sequential(
                torch.nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                torch.nn.BatchNorm2d(out_f),
                torch.nn.ReLU(),
                torch.nn.MaxPool2d(2)
            )
        self.features = torch.nn.Sequential(
            conv_block(1, 32), conv_block(32, 64), conv_block(64, 128),
            conv_block(128, 256), conv_block(256, 256)
        )
        self.embedding_head = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten(),
            torch.nn.Linear(256, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, embedding_dim)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.embedding_head(x)
        return F.normalize(x, p=2, dim=1)

def load_and_preprocess(path):
    waveform, sr = torchaudio.load(path)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
    if waveform.shape[0] > 1: waveform = waveform.mean(0, keepdim=True)
    
    mel_tf = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512)
    db_tf = torchaudio.transforms.AmplitudeToDB()
    spec = db_tf(mel_tf(waveform))
    spec = F.interpolate(spec.unsqueeze(0), size=(128, 128))
    spec = (spec - spec.mean()) / (spec.std() + 1e-7)
    return spec.to(DEVICE)

# --- BUILDING THE INDEX ---
def build_faiss_index():
    # 1. Inicijalizacija modela
    model = VoiceNetEmbedding(EMBEDDING_DIM).to(DEVICE)
    if not os.path.exists(MODEL_PATH):
        print(f"Baza znanja nije pronađena na {MODEL_PATH}!")
        return
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    # 2. Priprema FAISS indeksa
    index = faiss.IndexFlatL2(EMBEDDING_DIM)
    metadata = [] # Lista koja mapira index u bazi na info o fileu

    # 3. Iteracija kroz klase (isto kao tvoj folder_map)
    person_folders = sorted(glob.glob(os.path.join(DATASET_DIR, "*")))
    
    print(f"🚀 Krećem u izgradnju indeksa za {len(person_folders)} osoba...")

    all_embeddings = []
    
    with torch.inference_mode():
        for person_path in tqdm(person_folders, desc="Osobe"):
            person_id = os.path.basename(person_path)
            wav_files = glob.glob(os.path.join(person_path, "**/*.wav"), recursive=True)
            
            for f in wav_files:
                spec = load_and_preprocess(f)
                emb = model(spec).cpu().numpy()
                
                all_embeddings.append(emb)
                metadata.append({
                    "person_id": person_id,
                    "file_path": f
                })

    # 4. Dodavanje u FAISS (batch mode)
    if all_embeddings:
        final_embeddings = np.vstack(all_embeddings).astype('float32')
        index.add(final_embeddings)
        
        # 5. Spremanje na disk
        faiss.write_index(index, INDEX_NAME)
        with open(METADATA_NAME, 'wb') as f:
            pickle.dump(metadata, f)
            
        print(f"✅ Indeks uspješno spremljen! Broj zapisa: {index.ntotal}")
    else:
        print("Nema pronađenih podataka.")

# --- SEARCH FUNKCIJA ---
def search_voice(query_file, k=5000):
    # Učitaj resurse
    model = VoiceNetEmbedding(EMBEDDING_DIM).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    
    index = faiss.read_index(INDEX_NAME)
    with open(METADATA_NAME, 'rb') as f:
        metadata = pickle.load(f)

    # Procesiraj query
    spec = load_and_preprocess(query_file)
    with torch.inference_mode():
        query_emb = model(spec).cpu().numpy().astype('float32')

    # FAISS Search
    distances, indices = index.search(query_emb, k)

    print(f"\n🔍 Rezultati pretrage za: {os.path.basename(query_file)}")
    print("-" * 50)
    for i in range(k):
        idx = indices[0][i]
        dist = distances[0][i]
        if idx != -1:
            meta = metadata[idx]
            # S obzirom na Triplet Loss i normalizaciju, distanca < 0.6 je obično siguran match
            match_status = "✅ POGODAK" if dist < 0.6 else "❓ SLIČNO"
            print(f"{i+1}. ID: {meta['person_id']} | Dist: {dist:.4f} | {match_status}")
            print(f"   Path: {meta['file_path']}")

if __name__ == "__main__":
    # Prvo izgradi bazu
    build_faiss_index()
    
    # Primjer pretrage
    #test_file = "/home/antonio/Desktop/NMDU/dataset_voice/id10302/OMw6_X2IDvE/00001.wav"
    test_file = "/home/antonio/Desktop/NMDU/dataset_voice/id10302/VRCidrXwd1s/00013.wav"
    search_voice(test_file)

🚀 Krećem u izgradnju indeksa za 40 osoba...


Osobe: 100%|██████████| 40/40 [04:35<00:00,  6.90s/it]


✅ Indeks uspješno spremljen! Broj zapisa: 4874

🔍 Rezultati pretrage za: 00013.wav
--------------------------------------------------
1. ID: id10302 | Dist: 0.0000 | ✅ POGODAK
   Path: dataset_voice/id10302/VRCidrXwd1s/00013.wav
2. ID: id10302 | Dist: 0.0262 | ✅ POGODAK
   Path: dataset_voice/id10302/VRCidrXwd1s/00028.wav
3. ID: id10302 | Dist: 0.0302 | ✅ POGODAK
   Path: dataset_voice/id10302/VRCidrXwd1s/00010.wav
4. ID: id10302 | Dist: 0.0318 | ✅ POGODAK
   Path: dataset_voice/id10302/VRCidrXwd1s/00001.wav
5. ID: id10302 | Dist: 0.0345 | ✅ POGODAK
   Path: dataset_voice/id10302/VRCidrXwd1s/00022.wav
6. ID: id10302 | Dist: 0.0363 | ✅ POGODAK
   Path: dataset_voice/id10302/K2IuvkS1s38/00001.wav
7. ID: id10302 | Dist: 0.0364 | ✅ POGODAK
   Path: dataset_voice/id10302/VRCidrXwd1s/00005.wav
8. ID: id10302 | Dist: 0.0373 | ✅ POGODAK
   Path: dataset_voice/id10302/VRCidrXwd1s/00035.wav
9. ID: id10302 | Dist: 0.0376 | ✅ POGODAK
   Path: dataset_voice/id10302/VRCidrXwd1s/00008.wav
10. ID: id1

In [17]:
import torch
import torchaudio
import torch.nn.functional as F
import os
import glob
import faiss
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_curve, auc
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.calibration import calibration_curve
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore")

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_DIR          = "dataset_voice"
MODEL_PATH           = "triplet_run_2/best_triplet_model.pth"
SAMPLE_RATE          = 16000
EMBEDDING_DIM        = 128
DEVICE               = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Legacy single-threshold (kept for comparison plots) ──────────────────────
PRODUCTION_THRESHOLD = 0.10

# ── Open-set K1/K2 consensus parameters ──────────────────────────────────────
# K2: neighbourhood retrieval size  (how many FAISS hits to fetch)
# K1: minimum votes required to ACCEPT  (must be < K2)
# DIST_GATE: maximum allowed mean distance for winning class (hard reject above)
K2          = 9       # retrieve top-9 neighbors
K1          = 6       # require 6/9 votes for accept  (~67% consensus)
DIST_GATE   = 0.35    # distance ceiling even if votes pass

# ── FAR operating-point constraint ───────────────────────────────────────────
FAR_CONSTRAINT = 0.01

# ─── HELPERS ──────────────────────────────────────────────────────────────────
def get_run_folder():
    base, counter = "full_eval_report_", 1
    while os.path.exists(f"{base}{counter}"): counter += 1
    folder = f"{base}{counter}"
    os.makedirs(folder)
    return folder


# ─── MODEL ────────────────────────────────────────────────────────────────────
class VoiceNetEmbedding(torch.nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        def conv_block(in_f, out_f):
            return torch.nn.Sequential(
                torch.nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                torch.nn.BatchNorm2d(out_f), torch.nn.ReLU(), torch.nn.MaxPool2d(2)
            )
        self.features = torch.nn.Sequential(
            conv_block(1, 32), conv_block(32, 64), conv_block(64, 128),
            conv_block(128, 256), conv_block(256, 256)
        )
        self.embedding_head = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool2d((1, 1)), torch.nn.Flatten(),
            torch.nn.Linear(256, 256), torch.nn.ReLU(), torch.nn.Linear(256, embedding_dim)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.embedding_head(x)
        return F.normalize(x, p=2, dim=1)


def load_and_preprocess(path):
    try:
        waveform, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(0, keepdim=True)
        mel_tf = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        )
        db_tf = torchaudio.transforms.AmplitudeToDB()
        spec  = db_tf(mel_tf(waveform))
        spec  = F.interpolate(spec.unsqueeze(0), size=(128, 128))
        spec  = (spec - spec.mean()) / (spec.std() + 1e-7)
        return spec.to(DEVICE)
    except Exception:
        return None


# ═══════════════════════════════════════════════════════════════════════════════
#  OPEN-SET K1/K2 CONSENSUS DECISION ENGINE
# ═══════════════════════════════════════════════════════════════════════════════

def consensus_decision(distances: np.ndarray,
                        indices: np.ndarray,
                        gallery_labels: list,
                        k1: int,
                        dist_gate: float,
                        weighted: bool = True):
    """
    Open-set decision for a single query using K1/K2 consensus.

    Parameters
    ----------
    distances    : (K2,) L2 distances to K2 nearest gallery neighbors
    indices      : (K2,) gallery indices
    gallery_labels : flat list of all gallery labels
    k1           : minimum vote count to accept
    dist_gate    : maximum mean distance for the winning class (hard reject)
    weighted     : use inverse-distance weighting instead of raw vote counts

    Returns
    -------
    pred         : predicted identity str, or "unknown"
    confidence   : float in [0, 1] — vote share of winning class (entropy-based)
    winning_dist : mean L2 distance of winning-class neighbors
    vote_counts  : dict {label: weighted_votes}
    """
    k2 = len(indices)
    labels_k2 = [gallery_labels[idx] for idx in indices]

    if weighted:
        # Inverse-distance weights; clip to avoid division by zero
        eps     = 1e-6
        weights = 1.0 / (distances + eps)
        weights /= weights.sum()                      # normalize to sum=1
        vote_dict: dict[str, float] = {}
        for lbl, w in zip(labels_k2, weights):
            vote_dict[lbl] = vote_dict.get(lbl, 0.0) + w
    else:
        vote_dict = {lbl: labels_k2.count(lbl) for lbl in set(labels_k2)}

    # Winning class
    winning_label = max(vote_dict, key=vote_dict.get)
    winning_votes = vote_dict[winning_label]
    total_votes   = sum(vote_dict.values())

    # Raw vote count for K1 check (always integer count, not weighted)
    raw_count = labels_k2.count(winning_label)

    # Mean distance of winning-class neighbors
    winning_dists = [distances[j] for j, lbl in enumerate(labels_k2) if lbl == winning_label]
    winning_dist  = float(np.mean(winning_dists))

    # Confidence: vote share (weighted)
    confidence = float(winning_votes / total_votes) if total_votes > 0 else 0.0

    # ── Two-condition ACCEPT ──────────────────────────────────────────────────
    # Condition 1: raw vote majority ≥ K1
    # Condition 2: mean distance of winning class ≤ dist_gate
    if raw_count >= k1 and winning_dist <= dist_gate:
        return winning_label, confidence, winning_dist, vote_dict
    else:
        return "unknown", confidence, winning_dist, vote_dict


def run_consensus_on_fold(Q: np.ndarray,
                           G: np.ndarray,
                           query_labels: list,
                           gallery_labels: list,
                           k2: int,
                           k1: int,
                           dist_gate: float,
                           weighted: bool = True) -> list:
    """
    Runs K1/K2 consensus for all queries in one fold.
    Returns list of result dicts.
    """
    actual_k2 = min(k2, len(G))
    index = faiss.IndexFlatL2(EMBEDDING_DIM)
    index.add(G)
    D, I = index.search(Q, actual_k2)     # (N, k2) each

    results = []
    for i in range(len(query_labels)):
        pred, conf, w_dist, votes = consensus_decision(
            distances      = D[i],
            indices        = I[i],
            gallery_labels = gallery_labels,
            k1             = k1,
            dist_gate      = dist_gate,
            weighted       = weighted,
        )
        results.append({
            'true':       query_labels[i],
            'pred':       pred,
            'confidence': conf,
            'win_dist':   w_dist,
            'top1_dist':  float(D[i][0]),           # kept for legacy comparison
            'top1_label': gallery_labels[I[i][0]],  # kept for legacy comparison
            'votes':      votes,
        })
    return results


# ═══════════════════════════════════════════════════════════════════════════════
#  METRIC UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

def compute_eer(fpr, tpr, thresholds):
    fnr      = 1.0 - tpr
    idx      = np.argmin(np.abs(fpr - fnr))
    eer      = (fpr[idx] + fnr[idx]) / 2.0
    return eer, thresholds[idx]


def compute_det_curve(fpr, fnr):
    eps = 1e-6
    return np.clip(fpr, eps, 1 - eps), np.clip(fnr, eps, 1 - eps)


def separation_ratio(same_dist, diff_dist):
    if not same_dist or not diff_dist:
        return float('nan')
    mu_s, std_s = np.mean(same_dist), np.std(same_dist)
    mu_d, std_d = np.mean(diff_dist), np.std(diff_dist)
    denom = std_s + std_d
    return (mu_d - mu_s) / denom if denom > 0 else float('inf')


def operating_point_at_far(df_sweep, target_far=0.01):
    candidates = df_sweep[df_sweep['FAR'] <= target_far]
    if candidates.empty:
        return None
    return candidates.iloc[-1]


def platt_scale(scores, labels):
    from sklearn.linear_model import LogisticRegression
    X  = np.array(scores).reshape(-1, 1)
    y  = np.array(labels)
    lr = LogisticRegression()
    try:
        lr.fit(X, y)
        return lr.predict_proba(X)[:, 1]
    except Exception:
        return np.zeros(len(scores))


def threshold_stability(df_sweep, target_metric='F1', window=10):
    top_n = df_sweep.nlargest(window, target_metric)
    return top_n['Threshold'].mean(), top_n['Threshold'].std()


def sweep_k1_k2(processed_data, person_data, min_vids, k2_range, k1_fracs):
    """
    Grid search over K2 sizes and K1 fractions.
    Returns a DataFrame with FAR / FRR / F1 per (K2, K1, dist_gate) combination.
    Uses the same dist_gate as the global DIST_GATE for simplicity.
    """
    rows = []
    for k2_val in k2_range:
        for k1_frac in k1_fracs:
            k1_val = max(1, int(np.ceil(k1_frac * k2_val)))
            fold_results = []
            for fold in range(min_vids):
                gallery_embs, gallery_labels = [], []
                query_embs,   query_labels   = [], []
                for p_id, vids_embs in processed_data.items():
                    for i, emb_set in enumerate(vids_embs):
                        if i == fold:
                            query_embs.append(emb_set)
                            query_labels.extend([p_id] * len(emb_set))
                        else:
                            gallery_embs.append(emb_set)
                            gallery_labels.extend([p_id] * len(emb_set))
                G = np.vstack(gallery_embs)
                Q = np.vstack(query_embs)
                fold_results += run_consensus_on_fold(
                    Q, G, query_labels, gallery_labels,
                    k2=k2_val, k1=k1_val, dist_gate=DIST_GATE, weighted=True
                )
            # compute metrics
            y_true = [r['true'] for r in fold_results]
            y_pred = [r['pred'] for r in fold_results]
            known_mask = [r['pred'] != 'unknown' for r in fold_results]
            genuine_dists = [r['win_dist'] for r in fold_results if r['true'] == r['top1_label']]
            impostor_dists= [r['win_dist'] for r in fold_results if r['true'] != r['top1_label']]
            far = (sum(1 for r in fold_results if r['true'] != r['pred'] and r['pred'] != 'unknown')
                   / max(1, len([r for r in fold_results if r['true'] != r['top1_label']])))
            frr = (sum(1 for r in fold_results if r['pred'] == 'unknown' and r['true'] == r['top1_label'])
                   / max(1, len([r for r in fold_results if r['true'] == r['top1_label']])))
            _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
            rows.append({'K2': k2_val, 'K1': k1_val, 'K1_frac': k1_frac,
                         'FAR': far, 'FRR': frr, 'F1': f1})
    return pd.DataFrame(rows)


# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def run_full_evaluation():
    RUN_FOLDER = get_run_folder()
    print(f"🚀  Full evaluation (open-set K1/K2) → {RUN_FOLDER}")

    # ── Load model ────────────────────────────────────────────────────────────
    model = VoiceNetEmbedding(EMBEDDING_DIM).to(DEVICE)
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print(f"✅  Model loaded: {MODEL_PATH}")
    else:
        print(f"❌  Model not found: {MODEL_PATH}"); return
    model.eval()

    # ── Data extraction ───────────────────────────────────────────────────────
    person_ids  = sorted([d for d in os.listdir(DATASET_DIR)
                           if os.path.isdir(os.path.join(DATASET_DIR, d))])
    person_data = {}
    for p_id in person_ids:
        vids = sorted([v for v in glob.glob(os.path.join(DATASET_DIR, p_id, "*"))
                       if os.path.isdir(v)])
        if len(vids) > 1:
            person_data[p_id] = vids

    min_vids       = min(len(v) for v in person_data.values())
    processed_data = {p_id: [] for p_id in person_data}
    all_embs_for_vis, all_labels_for_vis = [], []
    latency_ms_list = []

    with torch.inference_mode():
        for p_id, vids in tqdm(person_data.items(), desc="🧬 Extraction"):
            for vid_path in vids[:min_vids]:
                wavs     = glob.glob(os.path.join(vid_path, "*.wav"))
                vid_embs = []
                for w in wavs:
                    spec = load_and_preprocess(w)
                    if spec is not None:
                        t0  = time.perf_counter()
                        emb = model(spec).cpu().numpy().astype('float32')
                        latency_ms_list.append((time.perf_counter() - t0) * 1000)
                        vid_embs.append(emb)
                        all_embs_for_vis.append(emb)
                        all_labels_for_vis.append(p_id)
                if vid_embs:
                    processed_data[p_id].append(np.vstack(vid_embs))

    # ── Cross-validation with K1/K2 consensus ────────────────────────────────
    all_results = []
    for fold in range(min_vids):
        gallery_embs, gallery_labels = [], []
        query_embs,   query_labels   = [], []

        for p_id, vids_embs in processed_data.items():
            for i, emb_set in enumerate(vids_embs):
                if i == fold:
                    query_embs.append(emb_set)
                    query_labels.extend([p_id] * len(emb_set))
                else:
                    gallery_embs.append(emb_set)
                    gallery_labels.extend([p_id] * len(emb_set))

        G = np.vstack(gallery_embs)
        Q = np.vstack(query_embs)

        fold_res = run_consensus_on_fold(
            Q, G, query_labels, gallery_labels,
            k2=K2, k1=K1, dist_gate=DIST_GATE, weighted=True
        )
        all_results.extend(fold_res)

    # ── Distance lists for analysis ───────────────────────────────────────────
    # "genuine" = top-1 label matches true label (regardless of consensus decision)
    same_dist = [r['top1_dist'] for r in all_results if r['true'] == r['top1_label']]
    diff_dist = [r['top1_dist'] for r in all_results if r['true'] != r['top1_label']]

    # ── Legacy threshold sweep (for comparison plots) ─────────────────────────
    thresholds = np.linspace(0.01, 1.0, 200)
    sweep_rows = []
    for t in thresholds:
        y_true, y_pred_t = [], []
        for res in all_results:
            y_true.append(res['true'])
            y_pred_t.append(res['top1_label'] if res['top1_dist'] < t else "unknown")
        acc          = accuracy_score(y_true, y_pred_t)
        p, r, f1, _  = precision_recall_fscore_support(y_true, y_pred_t, average='weighted', zero_division=0)
        far          = sum(1 for d in diff_dist if d < t) / max(1, len(diff_dist))
        frr          = sum(1 for d in same_dist if d >= t) / max(1, len(same_dist))
        sweep_rows.append({'Threshold': t, 'Accuracy': acc, 'Precision': p,
                           'Recall': r, 'F1': f1, 'FAR': far, 'FRR': frr})
    df_legacy = pd.DataFrame(sweep_rows)
    df_legacy.to_csv(f"{RUN_FOLDER}/legacy_threshold_sweep.csv", index=False)

    # ── Consensus system performance ──────────────────────────────────────────
    y_true_c  = [r['true']  for r in all_results]
    y_pred_c  = [r['pred']  for r in all_results]

    # FAR: impostor accepted as someone (pred != unknown AND pred != true)
    impostors      = [r for r in all_results if r['true'] != r['top1_label']]
    consensus_far  = sum(1 for r in impostors if r['pred'] != 'unknown') / max(1, len(impostors))
    # FRR: genuine rejected (pred == unknown AND top1 was correct class)
    genuines       = [r for r in all_results if r['true'] == r['top1_label']]
    consensus_frr  = sum(1 for r in genuines  if r['pred'] == 'unknown')  / max(1, len(genuines))
    consensus_acc  = accuracy_score(y_true_c, y_pred_c)
    _, _, consensus_f1, _ = precision_recall_fscore_support(
        y_true_c, y_pred_c, average='weighted', zero_division=0
    )
    unknown_rate = sum(1 for r in all_results if r['pred'] == 'unknown') / len(all_results)

    # ── ROC / AUC / EER (on top-1 distances, binary genuine/impostor) ─────────
    binary_labels = [1 if r['true'] == r['top1_label'] else 0 for r in all_results]
    scores_neg    = [-r['top1_dist'] for r in all_results]
    fpr, tpr, roc_thresh = roc_curve(binary_labels, scores_neg)
    roc_auc              = auc(fpr, tpr)
    eer, eer_threshold   = compute_eer(fpr, tpr, roc_thresh)

    # DET
    fnr = 1.0 - tpr
    det_fpr, det_fnr = compute_det_curve(fpr, fnr)

    # Separation ratio
    sep_ratio = separation_ratio(same_dist, diff_dist)

    # Operating point
    op_row = operating_point_at_far(df_legacy, target_far=FAR_CONSTRAINT)

    # Threshold stability
    stab_thresh, stab_std = threshold_stability(df_legacy, target_metric='F1', window=10)

    # Latency
    lat_mean = np.mean(latency_ms_list) if latency_ms_list else 0
    lat_p99  = np.percentile(latency_ms_list, 99) if latency_ms_list else 0

    # Calibration (on confidence scores)
    conf_scores     = [r['confidence'] for r in all_results]
    impostor_binary = [0 if r['true'] == r['top1_label'] else 1 for r in all_results]
    cal_probs       = platt_scale([r['top1_dist'] for r in all_results], impostor_binary)
    try:
        frac_pos, mean_pred = calibration_curve(impostor_binary, cal_probs, n_bins=10)
    except Exception:
        frac_pos, mean_pred = np.array([]), np.array([])

    # ── K1/K2 grid search ─────────────────────────────────────────────────────
    print("🔍  Running K1/K2 grid search …")
    df_grid = sweep_k1_k2(
        processed_data, person_data, min_vids,
        k2_range  = [5, 7, 9, 11, 15],
        k1_fracs  = [0.5, 0.6, 0.67, 0.75, 0.8]
    )
    df_grid.to_csv(f"{RUN_FOLDER}/k1k2_grid_search.csv", index=False)

    # ═══════════════════════════════════════════════════════════════════════════
    #  PLOTS
    # ═══════════════════════════════════════════════════════════════════════════
    DARK   = "#0f1117"
    GRID_C = "#1e2130"
    ACCENT = "#00e5ff"
    GREEN  = "#00ff9f"
    RED    = "#ff4757"
    ORANGE = "#ffa502"
    PURPLE = "#a29bfe"
    TEXT   = "#e8eaf6"
    plt.rcParams.update({
        "figure.facecolor": DARK,   "axes.facecolor": GRID_C,
        "axes.edgecolor":   "#333", "axes.labelcolor": TEXT,
        "xtick.color": TEXT,        "ytick.color": TEXT,
        "text.color":  TEXT,        "grid.color": "#2a2d3e",
        "legend.facecolor": GRID_C, "legend.edgecolor": "#444",
        "font.family": "monospace"
    })

    # ── P1: Embedding Clusters ────────────────────────────────────────────────
    X_vis = np.vstack(all_embs_for_vis)
    fig, axes = plt.subplots(1, 2, figsize=(20, 8), facecolor=DARK)
    fig.suptitle("Embedding Space Projections", color=TEXT, fontsize=16, fontweight='bold')
    unique_labels   = sorted(set(all_labels_for_vis))
    palette         = sns.color_palette("tab10", n_colors=len(unique_labels))
    label_to_color  = {l: palette[i] for i, l in enumerate(unique_labels)}
    colors          = [label_to_color[l] for l in all_labels_for_vis]
    pca_res         = PCA(n_components=2).fit_transform(X_vis)
    axes[0].scatter(pca_res[:,0], pca_res[:,1], c=colors, s=30, alpha=0.7, edgecolors='none')
    axes[0].set_title("PCA Projection", color=TEXT); axes[0].grid(True, alpha=0.2)
    safe_perp = min(30, max(1, X_vis.shape[0] - 1))
    try:
        tsne_res = TSNE(n_components=2, perplexity=safe_perp, init='pca',
                        learning_rate='auto').fit_transform(X_vis)
        axes[1].scatter(tsne_res[:,0], tsne_res[:,1], c=colors, s=30, alpha=0.7, edgecolors='none')
        axes[1].set_title(f"t-SNE (perp={safe_perp})", color=TEXT)
    except Exception as e:
        axes[1].text(0.5, 0.5, f"t-SNE Error:\n{e}", ha='center', va='center',
                     transform=axes[1].transAxes)
    axes[1].grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(f"{RUN_FOLDER}/clusters_projection.png", dpi=150, bbox_inches='tight')
    plt.close()

    # ── P2: ROC + AUC ─────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 7), facecolor=DARK)
    ax.plot(fpr, tpr, color=ACCENT, lw=2, label=f"ROC (AUC={roc_auc:.4f})")
    ax.plot([0,1],[0,1], 'k--', alpha=0.4, label="Chance")
    eer_fpr_idx = np.argmin(np.abs(fpr - eer))
    ax.scatter([fpr[eer_fpr_idx]], [tpr[eer_fpr_idx]], color=GREEN, s=80, zorder=5,
               label=f"EER ≈ {eer:.4f}")
    ax.set_xlabel("False Positive Rate (FAR)")
    ax.set_ylabel("True Positive Rate (1-FRR)")
    ax.set_title(f"ROC  |  AUC={roc_auc:.4f}  |  EER={eer:.4f}", fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(f"{RUN_FOLDER}/roc_auc.png", dpi=150, bbox_inches='tight'); plt.close()

    # ── P3: DET Curve ─────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 7), facecolor=DARK)
    ax.plot(det_fpr * 100, det_fnr * 100, color=RED, lw=2)
    ax.set_xscale('log'); ax.set_yscale('log')
    diag = np.linspace(1e-3, 100, 300)
    ax.plot(diag, diag, 'k--', alpha=0.3, label="EER line")
    ax.axvline(eer*100, color=GREEN, linestyle=':', label=f"EER={eer*100:.2f}%")
    ax.axhline(eer*100, color=GREEN, linestyle=':', alpha=0.6)
    ax.set_xlabel("FAR (%)"); ax.set_ylabel("FRR (%)")
    ax.set_title(f"DET Curve  |  EER≈{eer*100:.2f}%", fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.2, which='both')
    plt.tight_layout()
    plt.savefig(f"{RUN_FOLDER}/det_curve.png", dpi=150, bbox_inches='tight'); plt.close()

    # ── P4: Score Distributions ───────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), facecolor=DARK)
    sns.kdeplot(same_dist, ax=axes[0], label="Genuine",  fill=True, color=GREEN, alpha=0.5)
    sns.kdeplot(diff_dist, ax=axes[0], label="Impostor", fill=True, color=RED,   alpha=0.5)
    axes[0].axvline(PRODUCTION_THRESHOLD, color='white', linestyle='--',
                    label=f"Legacy T={PRODUCTION_THRESHOLD}")
    axes[0].axvline(DIST_GATE, color=ORANGE, linestyle=':',
                    label=f"Consensus dist_gate={DIST_GATE}")
    axes[0].axvline(-eer_threshold, color=PURPLE, linestyle='-.',
                    label=f"EER T≈{-eer_threshold:.3f}")
    axes[0].set_title(f"Score Distributions  |  Sep Ratio={sep_ratio:.3f}", fontweight='bold')
    axes[0].set_xlabel("L2 Distance"); axes[0].legend(); axes[0].grid(True, alpha=0.2)
    axes[1].hist(same_dist, bins=40, color=GREEN, alpha=0.6, label="Genuine",  density=True)
    axes[1].hist(diff_dist, bins=40, color=RED,   alpha=0.6, label="Impostor", density=True)
    axes[1].set_title("Score Histograms", fontweight='bold')
    axes[1].set_xlabel("L2 Distance"); axes[1].legend(); axes[1].grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(f"{RUN_FOLDER}/score_distributions.png", dpi=150, bbox_inches='tight'); plt.close()

    # ── P5: FAR/FRR (legacy threshold) + operating points ─────────────────────
    fig, ax = plt.subplots(figsize=(10, 6), facecolor=DARK)
    ax.plot(df_legacy['Threshold'], df_legacy['FAR'], color=RED,   lw=2, label='FAR (legacy)')
    ax.plot(df_legacy['Threshold'], df_legacy['FRR'], color=ACCENT, lw=2, label='FRR (legacy)')
    ax.axvline(PRODUCTION_THRESHOLD, color='white',  linestyle='--',
               label=f"Prod T={PRODUCTION_THRESHOLD}")
    ax.axvline(-eer_threshold, color=PURPLE, linestyle=':',
               label=f"EER T≈{-eer_threshold:.3f}")
    # Consensus operating points as horizontal lines
    ax.axhline(consensus_far, color=GREEN,  linestyle='-.', lw=1.5,
               label=f"Consensus FAR={consensus_far:.4f}")
    ax.axhline(consensus_frr, color=ORANGE, linestyle='-.', lw=1.5,
               label=f"Consensus FRR={consensus_frr:.4f}")
    if op_row is not None:
        ax.axvline(op_row['Threshold'], color=GREEN, linestyle='--', alpha=0.5,
                   label=f"Op.Point T={op_row['Threshold']:.3f}")
    ax.set_title("FAR vs FRR — Legacy vs Consensus", fontweight='bold')
    ax.set_xlabel("Threshold (legacy)"); ax.set_ylabel("Error Rate")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(f"{RUN_FOLDER}/security_analysis.png", dpi=150, bbox_inches='tight'); plt.close()

    # ── P6: Consensus Confusion Matrix ───────────────────────────────────────
    labels_cm = sorted(person_data.keys()) + ["unknown"]
    cm = confusion_matrix(y_true_c, y_pred_c, labels=labels_cm)
    fig, ax = plt.subplots(figsize=(14, 11), facecolor=DARK)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels_cm, yticklabels=labels_cm[:-1], ax=ax)
    ax.set_title(f"Consensus Confusion Matrix  (K2={K2}, K1={K1}, gate={DIST_GATE})", fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{RUN_FOLDER}/confusion_matrix.png", dpi=150, bbox_inches='tight'); plt.close()

    # ── P7: K1/K2 Grid Search Heatmap ────────────────────────────────────────
    for metric in ['FAR', 'FRR', 'F1']:
        pivot = df_grid.pivot(index='K1_frac', columns='K2', values=metric)
        fig, ax = plt.subplots(figsize=(10, 6), facecolor=DARK)
        cmap = 'RdYlGn' if metric == 'F1' else 'RdYlGn_r'
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap=cmap, ax=ax,
                    linewidths=0.5, linecolor='#333')
        ax.set_title(f"K1/K2 Grid Search — {metric}", fontweight='bold')
        ax.set_xlabel("K2 (neighbourhood size)")
        ax.set_ylabel("K1 fraction (consensus threshold)")
        plt.tight_layout()
        plt.savefig(f"{RUN_FOLDER}/k1k2_grid_{metric.lower()}.png", dpi=150, bbox_inches='tight')
        plt.close()

    # ── P8: Vote Confidence Distribution ─────────────────────────────────────
    correct_conf  = [r['confidence'] for r in all_results if r['pred'] == r['true'] and r['pred'] != 'unknown']
    wrong_conf    = [r['confidence'] for r in all_results if r['pred'] != r['true'] and r['pred'] != 'unknown']
    unknown_conf  = [r['confidence'] for r in all_results if r['pred'] == 'unknown']
    fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK)
    if correct_conf:  sns.kdeplot(correct_conf,  ax=ax, label="Correct accept",   fill=True, color=GREEN,  alpha=0.5)
    if wrong_conf:    sns.kdeplot(wrong_conf,     ax=ax, label="Wrong accept",     fill=True, color=RED,    alpha=0.5)
    if unknown_conf:  sns.kdeplot(unknown_conf,   ax=ax, label="Rejected (unknown)", fill=True, color=ORANGE, alpha=0.5)
    ax.set_xlabel("Vote Confidence (weighted share of winning class)")
    ax.set_title("Confidence Distribution by Decision Type", fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(f"{RUN_FOLDER}/confidence_distribution.png", dpi=150, bbox_inches='tight'); plt.close()

    # ── P9: Calibration ───────────────────────────────────────────────────────
    if len(frac_pos):
        fig, ax = plt.subplots(figsize=(7, 6), facecolor=DARK)
        ax.plot(mean_pred, frac_pos, 's-', color=ACCENT, lw=2, label="Platt-calibrated")
        ax.plot([0,1],[0,1], 'k--', alpha=0.5, label="Perfect calibration")
        ax.set_xlabel("Predicted probability (impostor)")
        ax.set_ylabel("Fraction of true impostors")
        ax.set_title("Calibration Curve (Platt Scaling)", fontweight='bold')
        ax.legend(); ax.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.savefig(f"{RUN_FOLDER}/calibration_curve.png", dpi=150, bbox_inches='tight'); plt.close()

    # ── P10: Latency ──────────────────────────────────────────────────────────
    if latency_ms_list:
        fig, ax = plt.subplots(figsize=(9, 5), facecolor=DARK)
        ax.hist(latency_ms_list, bins=40, color=ACCENT, alpha=0.8, edgecolor='none')
        ax.axvline(lat_mean, color=GREEN, linestyle='--', label=f"Mean={lat_mean:.2f}ms")
        ax.axvline(lat_p99,  color=RED,   linestyle=':',  label=f"P99={lat_p99:.2f}ms")
        ax.set_xlabel("Inference Latency (ms)"); ax.set_ylabel("Count")
        ax.set_title("Inference Latency Distribution", fontweight='bold')
        ax.legend(); ax.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.savefig(f"{RUN_FOLDER}/latency_distribution.png", dpi=150, bbox_inches='tight'); plt.close()

    # ── P11: Threshold Stability ───────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK)
    ax.plot(df_legacy['Threshold'], df_legacy['F1'],       color=ACCENT, lw=2,   label="F1")
    ax.plot(df_legacy['Threshold'], df_legacy['Accuracy'], color=GREEN,  lw=1.5, linestyle='--', label="Accuracy")
    ax.axvspan(stab_thresh - stab_std, stab_thresh + stab_std,
               alpha=0.15, color=ORANGE, label=f"Stability band (σ={stab_std:.4f})")
    ax.axvline(stab_thresh, color=ORANGE, linestyle=':', label=f"Best T zone≈{stab_thresh:.3f}")
    ax.axvline(PRODUCTION_THRESHOLD, color='white', linestyle='--', label=f"Prod T={PRODUCTION_THRESHOLD}")
    ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
    ax.set_title(f"Threshold Stability  σ={stab_std:.4f} → "
                 f"{'STABLE' if stab_std < 0.05 else 'UNSTABLE'}", fontweight='bold')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(f"{RUN_FOLDER}/threshold_stability.png", dpi=150, bbox_inches='tight'); plt.close()

    # ═══════════════════════════════════════════════════════════════════════════
    #  FINAL REPORT
    # ═══════════════════════════════════════════════════════════════════════════
    best_legacy = df_legacy.iloc[(df_legacy['Threshold'] - PRODUCTION_THRESHOLD).abs().argsort().iloc[0]]

    report_lines = [
        "=" * 65,
        f"  FULL EVALUATION REPORT (Open-Set K1/K2)  |  {RUN_FOLDER}",
        "=" * 65,
        "",
        "── Core Biometric Metrics (ROC/AUC/EER on top-1 distances) ──",
        f"  ROC AUC              : {roc_auc:.4f}",
        f"  EER                  : {eer*100:.2f}%  (at dist≈{-eer_threshold:.4f})",
        f"  Separation Ratio     : {sep_ratio:.3f}",
        "",
        f"── Legacy Threshold System  (T={PRODUCTION_THRESHOLD}) ──",
        f"  Accuracy             : {float(best_legacy['Accuracy']):.4f}",
        f"  FAR                  : {float(best_legacy['FAR']):.4f}",
        f"  FRR                  : {float(best_legacy['FRR']):.4f}",
        f"  F1                   : {float(best_legacy['F1']):.4f}",
        "",
        f"── Open-Set Consensus  (K2={K2}, K1={K1}, dist_gate={DIST_GATE}) ──",
        f"  FAR                  : {consensus_far:.4f}  ({'✅ better' if consensus_far < float(best_legacy['FAR']) else '⚠️ worse'} than legacy)",
        f"  FRR                  : {consensus_frr:.4f}  ({'✅ better' if consensus_frr < float(best_legacy['FRR']) else '⚠️ worse'} than legacy)",
        f"  Accuracy             : {consensus_acc:.4f}",
        f"  F1                   : {consensus_f1:.4f}",
        f"  Unknown Rate         : {unknown_rate*100:.1f}%  (samples rejected as unknown)",
        "",
        "── Operating Points ──",
    ]
    if op_row is not None:
        report_lines.append(
            f"  FAR≤{FAR_CONSTRAINT} operating point : T={op_row['Threshold']:.4f}, "
            f"FAR={op_row['FAR']:.4f}, FRR={op_row['FRR']:.4f}"
        )
    else:
        report_lines.append(f"  FAR≤{FAR_CONSTRAINT} operating point : not achievable in sweep")

    report_lines += [
        "",
        "── Operational Metrics ──",
        f"  Inference latency    : mean={lat_mean:.2f}ms, p99={lat_p99:.2f}ms",
        f"  Threshold stability  : best_zone≈{stab_thresh:.4f}, σ={stab_std:.4f}",
        "",
        "── Output Files ──",
        "  clusters_projection.png     roc_auc.png",
        "  det_curve.png               score_distributions.png",
        "  security_analysis.png       confusion_matrix.png",
        "  confidence_distribution.png calibration_curve.png",
        "  latency_distribution.png    threshold_stability.png",
        "  k1k2_grid_far.png           k1k2_grid_frr.png",
        "  k1k2_grid_f1.png            k1k2_grid_search.csv",
        "  legacy_threshold_sweep.csv  summary.csv",
        "=" * 65,
    ]

    report_text = "\n".join(report_lines)
    print("\n" + report_text)
    with open(f"{RUN_FOLDER}/report.txt", "w") as f:
        f.write(report_text)

    # Summary CSV
    summary = {
        "roc_auc": roc_auc, "eer": eer, "eer_threshold": -eer_threshold,
        "separation_ratio": sep_ratio,
        "legacy_threshold": PRODUCTION_THRESHOLD,
        "legacy_accuracy": float(best_legacy['Accuracy']),
        "legacy_far": float(best_legacy['FAR']),
        "legacy_frr": float(best_legacy['FRR']),
        "legacy_f1":  float(best_legacy['F1']),
        "consensus_k2": K2, "consensus_k1": K1, "consensus_dist_gate": DIST_GATE,
        "consensus_far": consensus_far, "consensus_frr": consensus_frr,
        "consensus_accuracy": consensus_acc, "consensus_f1": consensus_f1,
        "unknown_rate": unknown_rate,
        "lat_mean_ms": lat_mean, "lat_p99_ms": lat_p99,
        "stability_threshold": stab_thresh, "stability_std": stab_std,
    }
    if op_row is not None:
        summary.update({"op_threshold": float(op_row['Threshold']),
                         "op_far": float(op_row['FAR']), "op_frr": float(op_row['FRR'])})

    pd.DataFrame([summary]).to_csv(f"{RUN_FOLDER}/summary.csv", index=False)
    print(f"\n✅  Evaluation complete → {RUN_FOLDER}/")


if __name__ == "__main__":
    run_full_evaluation()



🚀  Full evaluation (open-set K1/K2) → full_eval_report_13
✅  Model loaded: triplet_run_2/best_triplet_model.pth


🧬 Extraction: 100%|██████████| 40/40 [02:15<00:00,  3.38s/it]


🔍  Running K1/K2 grid search …

  FULL EVALUATION REPORT (Open-Set K1/K2)  |  full_eval_report_13

── Core Biometric Metrics (ROC/AUC/EER on top-1 distances) ──
  ROC AUC              : 0.8652
  EER                  : 21.61%  (at dist≈0.0650)
  Separation Ratio     : 0.655

── Legacy Threshold System  (T=0.1) ──
  Accuracy             : 0.9014
  FAR                  : 0.3929
  FRR                  : 0.0883
  F1                   : 0.9420

── Open-Set Consensus  (K2=9, K1=6, dist_gate=0.35) ──
  FAR                  : 0.5714  (⚠️ worse than legacy)
  FRR                  : 0.0049  (✅ better than legacy)
  Accuracy             : 0.9859
  F1                   : 0.9906
  Unknown Rate         : 1.0%  (samples rejected as unknown)

── Operating Points ──
  FAR≤0.01 operating point : T=0.0398, FAR=0.0000, FRR=0.4857

── Operational Metrics ──
  Inference latency    : mean=15.92ms, p99=38.69ms
  Threshold stability  : best_zone≈0.8582, σ=0.0151

── Output Files ──
  clusters_projection.png    